In [1]:
import h5py
import numpy as np

H5_PATH = "../Dataset/spotify_dataset_compressed.h5"

def check_suitable_playlists(h5_path):
    with h5py.File(h5_path, 'r') as hf:
        print("1. 'Test' halmazba tartozó dalok kigyűjtése...")
        splits = hf['ml/split'][:]
        uris = hf['ml/track_uri'][:]
        
        # Byte stringek konvertálása, ha szükséges
        splits_str = np.array([s.decode('utf-8') if isinstance(s, bytes) else s for s in splits])
        uris_str = np.array([u.decode('utf-8') if isinstance(u, bytes) else u for u in uris])
        
        # Szűrjük ki a 'test' címkéjű dalok URI-jait
        test_uris = set(uris_str[splits_str == 'test'])
        print(f"   -> Összesen {len(test_uris)} dal van a CNN teszthalmazában.")

        print("\n2. Lejátszási listák adatainak betöltése...")
        pl_pids = hf['playlist_tracks/pid'][:]
        pl_uris = hf['playlist_tracks/track_uri'][:]
        pl_pos = hf['playlist_tracks/pos'][:]
        
        pl_uris_str = np.array([u.decode('utf-8') if isinstance(u, bytes) else u for u in pl_uris])

        print("3. Listák sorrendjének felépítése és szűrés...")
        # Csoportosítjuk a dalokat PID (lista ID) alapján
        # Egy szótárt hozunk létre: {pid: [(pos, uri), (pos, uri), ...]}
        playlists = {}
        for pid, uri, pos in zip(pl_pids, pl_uris_str, pl_pos):
            if pid not in playlists:
                playlists[pid] = []
            playlists[pid].append((pos, uri))

        suitable_playlists = 0
        total_playlists = len(playlists)

        # Végigmegyünk az összes listán
        for pid, tracks in playlists.items():
            # Sorba rendezzük a dalokat a pozíció (pos) alapján
            tracks.sort(key=lambda x: x[0])
            
            # Kinyerjük a legutolsó dalt
            last_track_uri = tracks[-1][1]
            
            # Megnézzük, hogy ez a legutolsó dal benne van-e a teszt halmazunkban
            if last_track_uri in test_uris:
                suitable_playlists += 1

        print(f"\n✅ Eredmény:")
        print(f"   Összes lejátszási lista: {total_playlists:,}")
        print(f"   Megfelelő listák száma (ahol az utolsó dal a 'test' halmazban van): {suitable_playlists:,}")

        return suitable_playlists

# Futtatás
if __name__ == "__main__":
    check_suitable_playlists(H5_PATH)

1. 'Test' halmazba tartozó dalok kigyűjtése...
   -> Összesen 2693 dal van a CNN teszthalmazában.

2. Lejátszási listák adatainak betöltése...
3. Listák sorrendjének felépítése és szűrés...

✅ Eredmény:
   Összes lejátszási lista: 1,000,000
   Megfelelő listák száma (ahol az utolsó dal a 'test' halmazban van): 50,251


# Még régebbi modell kiértékelés

In [5]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import faiss
import os
import gc
from tqdm.auto import tqdm

# --- 1. ÚTVONALAK ÉS PARAMÉTEREK ---
INPUT_PATH = os.path.join("../Models")
TEST_DATA_PATH = os.path.join("../Models/test_pids.npy")
SAVE_MODEL_PATH = os.path.join("../Models/best_sasrec_hybrid_fine_tuning_v02.weights.h5")

MAX_LEN = 50
D_MODEL = 400
TOP_K = 50

print("📂 Teszt adatok betöltése...")
test_playlists = np.load(TEST_DATA_PATH, allow_pickle=True)

# --- 2. MODELL ARCHITEKTÚRA (Egyeznie kell a tanítással!) ---
def create_model(vocab_size):
    inputs = layers.Input(shape=(MAX_LEN,))
    # Itt is trainable=True kell, hogy be tudja tölteni az új súlyokat!
    x = layers.Embedding(vocab_size, D_MODEL, mask_zero=True,
                         trainable=True, name="hybrid_embedding")(inputs)
    x += layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")(tf.range(MAX_LEN)[tf.newaxis, :])
    for i in range(2):
        attn = layers.MultiHeadAttention(4, D_MODEL, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))
    outputs = layers.Dense(D_MODEL, dtype='float32', name="final_projection")(x)
    return tf.keras.Model(inputs, outputs)

# --- 3. ÚJ SÚLYOK BETÖLTÉSE ---
print("🏗️ Modell felépítése és a 3.1GB-os súlyfájl betöltése...")
# Mivel nem tudjuk fejből a vocab_size-t, megnézzük a test fájlból a max ID-t, vagy beírjuk a 372k-t
# Feltételezve, hogy emlékszel a pontos méretre (pl. 372000 körüli érték), vagy betöltöd a régi npy-t csak a shape miatt:
dummy_hybrid = np.load(os.path.join(INPUT_PATH, "hybrid_embedding_matrix.npy"), mmap_mode='r')
vocab_size = dummy_hybrid.shape[0]

model = create_model(vocab_size)
model.load_weights(SAVE_MODEL_PATH)

# --- 4. AZ ÚJ VEKTORTÉR KINYERÉSE A MODELLBŐL ---
print("⚡ Új, finomhangolt hibrid vektorok kinyerése a modellből...")
# EZ A MÁGIA! Kiszívjuk a modellből azokat a vektorokat, amiket átrendezett.
updated_embeddings = model.get_layer("hybrid_embedding").get_weights()[0]

print("🏗️ FAISS index építése az ÚJ vektorokkal...")
faiss_embeddings = updated_embeddings.copy().astype('float32')
faiss.normalize_L2(faiss_embeddings)
index = faiss.IndexFlatIP(D_MODEL)
index.add(faiss_embeddings)

# --- 5. FAIR KIÉRTÉKELŐ (Szűréssel) ---
def run_evaluation_fair(playlists, k=50):
    hits = 0
    ndcg = 0
    map_score = 0
    total = 0
    
    print(f"🧐 Fair kiértékelés indítása {len(playlists):,} listán...")
    
    for pl in tqdm(playlists, desc="Evaluating"):
        if len(pl) < 2: continue
        
        context_indices = pl[:-1] 
        target_id = pl[-1]
        
        input_seq = context_indices[-MAX_LEN:]
        pad_len = MAX_LEN - len(input_seq)
        input_padded = np.array([[0] * pad_len + input_seq])
        
        # Predikció
        pred_vectors = model.predict(input_padded, verbose=0)
        target_pred_vector = pred_vectors[0, -1, :].reshape(1, -1).astype('float32')
        
        # FAISS keresés - Többet kérünk, hogy legyen miből szűrni!
        faiss.normalize_L2(target_pred_vector)
        _, raw_top_indices = index.search(target_pred_vector, k + len(context_indices))
        raw_top_indices = raw_top_indices[0]
        
        # --- SZŰRÉS: Csak az új dalokat hagyjuk meg ---
        recommendations = []
        for idx in raw_top_indices:
            if idx not in context_indices:
                recommendations.append(idx)
            if len(recommendations) == k:
                break
        
        # --- METRIKÁK ---
        total += 1
        if target_id in recommendations:
            hits += 1
            rank = recommendations.index(target_id) + 1
            ndcg += 1 / np.log2(rank + 1)
            map_score += 1 / rank

    final_hr = hits / total
    final_ndcg = ndcg / total
    final_map = map_score / total

    print("\n" + "="*35)
    print(f"🏆 FAIR JELENTÉS (Top {k}) - FINE TUNED")
    print("-" * 35)
    print(f"✨ Hit Rate:  {final_hr:.4%}")
    print(f"📈 NDCG:      {final_ndcg:.4%}")
    print(f"🎯 MAP:       {final_map:.4%}")
    print("="*35)

# Indítás (Egyelőre a teljes teszthalmazon)
run_evaluation_fair(test_playlists[:1000], k=TOP_K)

📂 Teszt adatok betöltése...
🏗️ Modell felépítése és a 3.1GB-os súlyfájl betöltése...
⚡ Új, finomhangolt hibrid vektorok kinyerése a modellből...
🏗️ FAISS index építése az ÚJ vektorokkal...
🧐 Fair kiértékelés indítása 1,000 listán...


Evaluating:   0%|          | 0/1000 [00:00<?, ?it/s]


🏆 FAIR JELENTÉS (Top 50) - FINE TUNED
-----------------------------------
✨ Hit Rate:  3.8000%
📈 NDCG:      1.1079%
🎯 MAP:       0.4915%


# Régebbi verzió kiértékelés

In [11]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import faiss
import os
import gc
from tqdm.auto import tqdm

# ==========================================
# 1. ÚTVONALAK (Állítsd be a saját gépedhez!)
# ==========================================
BASE_DIR = "../Models" # Írd át arra a mappára, ahol a fájljaid vannak!

HYBRID_MATRIX_PATH = os.path.join(BASE_DIR, "hybrid_embedding_matrix.npy")
TEST_DATA_PATH = os.path.join(BASE_DIR, "test_pids.npy")
WEIGHTS_PATH = os.path.join(BASE_DIR, "best_sasrec_bpr.weights.h5") # A Kaggle-ről letöltött fájl!

# ==========================================
# 2. HIPERPARAMÉTEREK
# ==========================================
MAX_LEN = 50
D_MODEL = 400

print("📂 Adatok betöltése...")
# A régi mátrixot csak az inicializálás és a vocab_size miatt töltjük be
hybrid_weights = np.load(HYBRID_MATRIX_PATH).astype('float32')
vocab_size = hybrid_weights.shape[0]

test_playlists = np.load(TEST_DATA_PATH, allow_pickle=True)

# ==========================================
# 3. A BPR MODELL ARCHITEKTÚRÁJA (A betöltéshez)
# ==========================================
def create_bpr_model(vocab_size, init_weights):
    seq_inputs = layers.Input(shape=(MAX_LEN,), name="seq_inputs")
    pos_inputs = layers.Input(shape=(MAX_LEN,), name="pos_inputs")
    neg_inputs = layers.Input(shape=(MAX_LEN,), name="neg_inputs")

    item_embedding = layers.Embedding(
        vocab_size, D_MODEL, mask_zero=True,
        embeddings_initializer=tf.keras.initializers.Constant(init_weights),
        trainable=True, name="hybrid_embedding"
    )
    pos_embedding = layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")

    x = item_embedding(seq_inputs)
    x += pos_embedding(tf.range(MAX_LEN)[tf.newaxis, :])
    
    for i in range(2):
        attn = layers.MultiHeadAttention(4, D_MODEL, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))
        
    seq_states = layers.Dense(D_MODEL, name="final_projection")(x)

    pos_emb = item_embedding(pos_inputs)
    neg_emb = item_embedding(neg_inputs)

    def dot_and_cast(tensors):
        seq, emb = tensors
        scores = tf.reduce_sum(seq * emb, axis=-1, keepdims=True)
        return tf.cast(scores, tf.float32)

    dot_layer = layers.Lambda(dot_and_cast, name="dot_scoring")
    
    pos_scores = dot_layer([seq_states, pos_emb])
    neg_scores = dot_layer([seq_states, neg_emb])

    outputs = layers.Concatenate(axis=-1, name="concat_scores")([pos_scores, neg_scores])
    return tf.keras.Model(inputs=[seq_inputs, pos_inputs, neg_inputs], outputs=outputs)

print("🏗️ Modell felépítése és súlyok betöltése...")
model = create_bpr_model(vocab_size, hybrid_weights)
model.load_weights(WEIGHTS_PATH)
print("✅ Kaggle súlyok sikeresen betöltve!")

# Memória takarítás (a régi mátrix már nem kell)
del hybrid_weights
gc.collect()

# ==========================================
# 4. INFERENCIA MODELL KIVÁGÁSA (Keras 3 fix)
# ==========================================
print("✂️ Inferencia modell kivágása a jósláshoz...")

# JAVÍTÁS: A model.inputs lista 0. eleme a seq_inputs!
inf_inputs = model.inputs[0] 
inf_outputs = model.get_layer("final_projection").output

inference_model = tf.keras.Model(inputs=inf_inputs, outputs=inf_outputs)
print("✅ Inferencia modell kész!")

# ==========================================
# 5. AZ ÚJ MÁGNESES VEKTORTÉR KINYERÉSE & FAISS
# ==========================================
print("⚡ Új, finomhangolt hibrid vektorok kinyerése a modellből...")
updated_embeddings = model.get_layer("hybrid_embedding").get_weights()[0]

print("🏗️ FAISS index építése...")
faiss_embeddings = updated_embeddings.copy().astype('float32')
faiss.normalize_L2(faiss_embeddings)
index = faiss.IndexFlatIP(D_MODEL)
index.add(faiss_embeddings)
print(f"📊 FAISS Indexben tárolt dalok száma: {index.ntotal}")

# ==========================================
# 6. MULTI-K FAIR KIÉRTÉKELŐ (Szűréssel)
# ==========================================
def run_evaluation_multi_k(playlists, k_list=[10, 50, 100]):
    max_k = max(k_list)
    
    hits = {k: 0 for k in k_list}
    ndcg = {k: 0 for k in k_list}
    map_score = {k: 0 for k in k_list}
    total = 0
    
    print(f"\n🧐 Fair kiértékelés indítása {len(playlists):,} listán...")
    
    for pl in tqdm(playlists, desc="Evaluating"):
        if len(pl) < 2: continue
        
        context_indices = pl[:-1] 
        target_id = pl[-1]
        
        input_seq = context_indices[-MAX_LEN:]
        pad_len = MAX_LEN - len(input_seq)
        input_padded = np.array([[0] * pad_len + input_seq])
        
        # PREDICITON AZ ÚJ INFERENCIA MODELLAL
        pred_vectors = inference_model.predict(input_padded, verbose=0)
        target_pred_vector = pred_vectors[0, -1, :].reshape(1, -1).astype('float32')
        
        # FAISS Keresés
        faiss.normalize_L2(target_pred_vector)
        _, raw_top_indices = index.search(target_pred_vector, max_k + len(context_indices))
        raw_top_indices = raw_top_indices[0]
        
        # --- SZŰRÉS: Ne ajánljuk azt, amit már hallgatott ---
        recommendations = []
        for idx in raw_top_indices:
            if idx not in context_indices:
                recommendations.append(idx)
            if len(recommendations) == max_k:
                break
        
        total += 1
        
        # --- METRIKÁK SZÁMÍTÁSA ---
        if target_id in recommendations:
            rank = recommendations.index(target_id) + 1
            for k in k_list:
                if rank <= k:
                    hits[k] += 1
                    ndcg[k] += 1 / np.log2(rank + 1)
                    map_score[k] += 1 / rank

    print("\n" + "🔥"*15 + " EREDMÉNYEK " + "🔥"*15)
    print("🏆 BPR (MÁGNESES) SASREC - MULTI-K JELENTÉS")
    print("="*45)
    for k in k_list:
        print(f"--- Top {k} Ajánlás ---")
        print(f"✨ Hit Rate:  {hits[k]/total:.4%}")
        print(f"📈 NDCG:      {ndcg[k]/total:.4%}")
        print(f"🎯 MAP:       {map_score[k]/total:.4%}\n")
    print("="*45)

# Indítás (a :: tesztelésnél kiveheted, ha csak egy részén akarod próbálni először, pl. test_playlists[:1000])
run_evaluation_multi_k(test_playlists[::10], k_list=[10, 20, 50, 100])

📂 Adatok betöltése...
🏗️ Modell felépítése és súlyok betöltése...
✅ Kaggle súlyok sikeresen betöltve!
✂️ Inferencia modell kivágása a jósláshoz...
✅ Inferencia modell kész!
⚡ Új, finomhangolt hibrid vektorok kinyerése a modellből...
🏗️ FAISS index építése...
📊 FAISS Indexben tárolt dalok száma: 372433

🧐 Fair kiértékelés indítása 9,918 listán...


Evaluating:   0%|          | 0/9918 [00:00<?, ?it/s]


🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥 EREDMÉNYEK 🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
🏆 BPR (MÁGNESES) SASREC - MULTI-K JELENTÉS
--- Top 10 Ajánlás ---
✨ Hit Rate:  5.3539%
📈 NDCG:      2.7968%
🎯 MAP:       2.0304%

--- Top 20 Ajánlás ---
✨ Hit Rate:  8.2678%
📈 NDCG:      3.5282%
🎯 MAP:       2.2283%

--- Top 50 Ajánlás ---
✨ Hit Rate:  14.0250%
📈 NDCG:      4.6666%
🎯 MAP:       2.4095%

--- Top 100 Ajánlás ---
✨ Hit Rate:  20.5687%
📈 NDCG:      5.7238%
🎯 MAP:       2.5013%



# Kiértékelés: Hibrid téren tanítás (trainable=True) és hibrid téren kiértékelés

In [15]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import faiss
import os
import gc
from tqdm.auto import tqdm

# ==========================================
# 1. ÚTVONALAK (Állítsd be a saját gépedhez!)
# ==========================================
BASE_DIR = "../Models" 

HYBRID_MATRIX_PATH = os.path.join(BASE_DIR, "hybrid_embedding_matrix.npy")
TEST_DATA_PATH = os.path.join(BASE_DIR, "test_pids.npy")

# FIGYELEM: Az új, nagy modell súlyfájlja!
WEIGHTS_PATH = os.path.join(BASE_DIR, "best_sasrec_bpr_large_hard.weights.h5") 

# ==========================================
# 2. HIPERPARAMÉTEREK
# ==========================================
MAX_LEN = 50
D_MODEL = 400

print("📂 Adatok betöltése...")
hybrid_weights = np.load(HYBRID_MATRIX_PATH).astype('float32')
vocab_size = hybrid_weights.shape[0]
test_playlists = np.load(TEST_DATA_PATH, allow_pickle=True)

# ==========================================
# 3. AZ ÚJ, NAGY ARCHITEKTÚRA (A betöltéshez)
# ==========================================
def create_large_bpr_model(vocab_size, init_weights):
    seq_inputs = layers.Input(shape=(MAX_LEN,), name="seq_inputs")
    pos_inputs = layers.Input(shape=(MAX_LEN,), name="pos_inputs")
    neg_inputs = layers.Input(shape=(MAX_LEN,), name="neg_inputs")

    item_embedding = layers.Embedding(
        vocab_size, D_MODEL, mask_zero=True,
        embeddings_initializer=tf.keras.initializers.Constant(init_weights),
        trainable=True, name="hybrid_embedding"
    )
    pos_embedding = layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")

    x = item_embedding(seq_inputs)
    x += pos_embedding(tf.range(MAX_LEN)[tf.newaxis, :])
    
    # FIGYELEM: Itt a 3 blokk és a 8 fej!
    for i in range(3): 
        attn = layers.MultiHeadAttention(num_heads=8, key_dim=D_MODEL//8, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))
        
    seq_states = layers.Dense(D_MODEL, name="final_projection")(x)

    pos_emb = item_embedding(pos_inputs)
    neg_emb = item_embedding(neg_inputs)

    def dot_and_cast(tensors):
        seq, emb = tensors
        scores = tf.reduce_sum(seq * emb, axis=-1, keepdims=True)
        return tf.cast(scores, tf.float32)

    dot_layer = layers.Lambda(dot_and_cast, name="dot_scoring")
    
    pos_scores = dot_layer([seq_states, pos_emb])
    neg_scores = dot_layer([seq_states, neg_emb])

    outputs = layers.Concatenate(axis=-1, name="concat_scores")([pos_scores, neg_scores])
    return tf.keras.Model(inputs=[seq_inputs, pos_inputs, neg_inputs], outputs=outputs)

print("🏗️ Modell felépítése és súlyok betöltése...")
model = create_large_bpr_model(vocab_size, hybrid_weights)
model.load_weights(WEIGHTS_PATH)
print("✅ Kaggle V3 súlyok sikeresen betöltve!")

del hybrid_weights
gc.collect()

# ==========================================
# 4. INFERENCIA MODELL KIVÁGÁSA (Keras 3 fix)
# ==========================================
print("✂️ Inferencia modell kivágása a jósláshoz...")
inf_inputs = model.inputs[0] # A szigorú Keras 3 bemenet
inf_outputs = model.get_layer("final_projection").output
inference_model = tf.keras.Model(inputs=inf_inputs, outputs=inf_outputs)

# ==========================================
# 5. AZ ÚJ MÁGNESES VEKTORTÉR KINYERÉSE & FAISS
# ==========================================
print("⚡ Finomhangolt hibrid vektorok kinyerése a modellből...")
updated_embeddings = model.get_layer("hybrid_embedding").get_weights()[0]

print("🏗️ FAISS index építése...")
faiss_embeddings = updated_embeddings.copy().astype('float32')
faiss.normalize_L2(faiss_embeddings)
index = faiss.IndexFlatIP(D_MODEL)
index.add(faiss_embeddings)
print(f"📊 FAISS Indexben tárolt dalok száma: {index.ntotal}")

# ==========================================
# 6. MULTI-K FAIR KIÉRTÉKELŐ
# ==========================================
def run_evaluation_multi_k(playlists, k_list=[10, 50, 100]):
    max_k = max(k_list)
    hits = {k: 0 for k in k_list}
    ndcg = {k: 0 for k in k_list}
    map_score = {k: 0 for k in k_list}
    total = 0
    
    print(f"\n🧐 Fair kiértékelés indítása {len(playlists):,} listán...")
    
    for pl in tqdm(playlists, desc="Evaluating"):
        if len(pl) < 2: continue
        
        context_indices = pl[:-1] 
        target_id = pl[-1]
        
        input_seq = context_indices[-MAX_LEN:]
        pad_len = MAX_LEN - len(input_seq)
        input_padded = np.array([[0] * pad_len + input_seq])
        
        pred_vectors = inference_model.predict(input_padded, verbose=0)
        target_pred_vector = pred_vectors[0, -1, :].reshape(1, -1).astype('float32')
        
        faiss.normalize_L2(target_pred_vector)
        _, raw_top_indices = index.search(target_pred_vector, max_k + len(context_indices))
        raw_top_indices = raw_top_indices[0]
        
        recommendations = []
        for idx in raw_top_indices:
            if idx not in context_indices:
                recommendations.append(idx)
            if len(recommendations) == max_k:
                break
        
        total += 1
        
        if target_id in recommendations:
            rank = recommendations.index(target_id) + 1
            for k in k_list:
                if rank <= k:
                    hits[k] += 1
                    ndcg[k] += 1 / np.log2(rank + 1)
                    map_score[k] += 1 / rank

    print("\n" + "🔥"*15 + " EREDMÉNYEK (V3 LARGE) " + "🔥"*15)
    for k in k_list:
        print(f"--- Top {k} Ajánlás ---")
        print(f"✨ Hit Rate:  {hits[k]/total:.4%}")
        print(f"📈 NDCG:      {ndcg[k]/total:.4%}")
        print(f"🎯 MAP:       {map_score[k]/total:.4%}\n")
    print("="*45)

# Indítás a teljes teszthalmazon!
run_evaluation_multi_k(test_playlists[::10], k_list=[10, 20, 50, 100, 500])

📂 Adatok betöltése...
🏗️ Modell felépítése és súlyok betöltése...
✅ Kaggle V3 súlyok sikeresen betöltve!
✂️ Inferencia modell kivágása a jósláshoz...
⚡ Finomhangolt hibrid vektorok kinyerése a modellből...
🏗️ FAISS index építése...
📊 FAISS Indexben tárolt dalok száma: 372433

🧐 Fair kiértékelés indítása 9,918 listán...


Evaluating:   0%|          | 0/9918 [00:00<?, ?it/s]


🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥 EREDMÉNYEK (V3 LARGE) 🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
--- Top 10 Ajánlás ---
✨ Hit Rate:  6.7251%
📈 NDCG:      3.5492%
🎯 MAP:       2.5898%

--- Top 20 Ajánlás ---
✨ Hit Rate:  9.7701%
📈 NDCG:      4.3171%
🎯 MAP:       2.7997%

--- Top 50 Ajánlás ---
✨ Hit Rate:  16.0617%
📈 NDCG:      5.5597%
🎯 MAP:       2.9969%

--- Top 100 Ajánlás ---
✨ Hit Rate:  21.6475%
📈 NDCG:      6.4630%
🎯 MAP:       3.0756%

--- Top 500 Ajánlás ---
✨ Hit Rate:  39.5039%
📈 NDCG:      8.7269%
🎯 MAP:       3.1568%



# Kiértékelés: W2V téren tanítás (trainable=True) és w2v téren kiértékelés

In [19]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import faiss
import os
import gc
from tqdm.auto import tqdm

# ==========================================
# 1. ÚTVONALAK (Állítsd be a saját gépedhez!)
# ==========================================
BASE_DIR = "../Models" 

HYBRID_MATRIX_PATH = os.path.join(BASE_DIR, "word2vec_matrix.npy")
TEST_DATA_PATH = os.path.join(BASE_DIR, "test_pids.npy")
WEIGHTS_PATH = os.path.join(BASE_DIR, "best_sasrec_large_hard_w2v_only.weights.h5")

# ==========================================
# 2. HIPERPARAMÉTEREK & ADATBETÖLTÉS
# ==========================================
MAX_LEN = 50

print("📂 Adatok betöltése...")
w2v_weights = np.load(HYBRID_MATRIX_PATH).astype('float32')
vocab_size = w2v_weights.shape[0]
D_MODEL = w2v_weights.shape[1]  # Dinamikus dimenzió!
test_playlists = np.load(TEST_DATA_PATH, allow_pickle=True)

# ==========================================
# 3. AZ ÚJ, NAGY ARCHITEKTÚRA
# ==========================================
def create_large_bpr_model(vocab_size, init_weights):
    seq_inputs = layers.Input(shape=(MAX_LEN,), name="seq_inputs")
    pos_inputs = layers.Input(shape=(MAX_LEN,), name="pos_inputs")
    neg_inputs = layers.Input(shape=(MAX_LEN,), name="neg_inputs")

    item_embedding = layers.Embedding(
        vocab_size, D_MODEL, mask_zero=True,
        embeddings_initializer=tf.keras.initializers.Constant(init_weights),
        trainable=True, name="w2v_embedding" # <--- JAVÍTVA!
    )
    pos_embedding = layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")

    x = item_embedding(seq_inputs)
    x += pos_embedding(tf.range(MAX_LEN)[tf.newaxis, :])
    
    for i in range(3): 
        attn = layers.MultiHeadAttention(num_heads=8, key_dim=D_MODEL//8, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))
        
    seq_states = layers.Dense(D_MODEL, name="final_projection")(x)

    pos_emb = item_embedding(pos_inputs)
    neg_emb = item_embedding(neg_inputs)

    def dot_and_cast(tensors):
        seq, emb = tensors
        scores = tf.reduce_sum(seq * emb, axis=-1, keepdims=True)
        return tf.cast(scores, tf.float32)

    dot_layer = layers.Lambda(dot_and_cast, name="dot_scoring")
    
    pos_scores = dot_layer([seq_states, pos_emb])
    neg_scores = dot_layer([seq_states, neg_emb])

    outputs = layers.Concatenate(axis=-1, name="concat_scores")([pos_scores, neg_scores])
    return tf.keras.Model(inputs=[seq_inputs, pos_inputs, neg_inputs], outputs=outputs)

print("🏗️ Modell felépítése és súlyok betöltése...")
model = create_large_bpr_model(vocab_size, w2v_weights)
model.load_weights(WEIGHTS_PATH)
print("✅ Kaggle W2V súlyok sikeresen betöltve!")

del w2v_weights
gc.collect()

# ==========================================
# 4. INFERENCIA & FAISS (JAVÍTVA)
# ==========================================
print("✂️ Inferencia modell kivágása a jósláshoz...")
inf_inputs = model.inputs[0] 
inf_outputs = model.get_layer("final_projection").output
inference_model = tf.keras.Model(inputs=inf_inputs, outputs=inf_outputs)

print("⚡ Finomhangolt W2V vektorok kinyerése a modellből...")
# <--- JAVÍTVA A RÉTEGNÉV!
updated_embeddings = model.get_layer("w2v_embedding").get_weights()[0] 

print("🏗️ FAISS index építése...")
faiss_embeddings = updated_embeddings.copy().astype('float32')
faiss.normalize_L2(faiss_embeddings)
index = faiss.IndexFlatIP(D_MODEL)
index.add(faiss_embeddings)
print(f"📊 FAISS Indexben tárolt dalok száma: {index.ntotal}")

# ==========================================
# 6. MULTI-K FAIR KIÉRTÉKELŐ (+ CLICKS)
# ==========================================
def run_evaluation_multi_k(playlists, k_list=[10, 50, 100]):
    max_k = max(k_list)
    hits = {k: 0 for k in k_list}
    ndcg = {k: 0 for k in k_list}
    map_score = {k: 0 for k in k_list}
    total_clicks = 0
    total = 0
    
    print(f"\n🧐 Fair kiértékelés indítása {len(playlists):,} listán...")
    
    for pl in tqdm(playlists, desc="Evaluating"):
        if len(pl) < 2: continue
        
        context_indices = pl[:-1] 
        target_id = pl[-1]
        
        input_seq = context_indices[-MAX_LEN:]
        pad_len = MAX_LEN - len(input_seq)
        input_padded = np.array([[0] * pad_len + input_seq])
        
        pred_vectors = inference_model.predict(input_padded, verbose=0)
        target_pred_vector = pred_vectors[0, -1, :].reshape(1, -1).astype('float32')
        
        faiss.normalize_L2(target_pred_vector)
        _, raw_top_indices = index.search(target_pred_vector, max_k + len(context_indices))
        raw_top_indices = raw_top_indices[0]
        
        recommendations = []
        for idx in raw_top_indices:
            if idx not in context_indices:
                recommendations.append(idx)
            if len(recommendations) == max_k:
                break
        
        total += 1
        
        # --- METRIKÁK ÉS CLICKS SZÁMÍTÁSA ---
        if target_id in recommendations:
            rank = recommendations.index(target_id) + 1
            current_clicks = (rank - 1) // 10  # RecSys Clicks matek
            
            for k in k_list:
                if rank <= k:
                    hits[k] += 1
                    ndcg[k] += 1 / np.log2(rank + 1)
                    map_score[k] += 1 / rank
        else:
            current_clicks = 51 # RecSys maximális büntetés
            
        total_clicks += current_clicks

    print("\n" + "🔥"*15 + " EREDMÉNYEK (TISZTA W2V) " + "🔥"*15)
    for k in k_list:
        print(f"--- Top {k} Ajánlás ---")
        print(f"✨ Hit Rate:  {hits[k]/total:.4%}")
        print(f"📈 NDCG:      {ndcg[k]/total:.4%}")
        print(f"🎯 MAP:       {map_score[k]/total:.4%}\n")
    print("="*45)
    # Az átlagos kattintásszám:
    print(f"🖱️ Átlagos RecSys Clicks: {total_clicks/total:.4f}")
    print("="*45)

# Indítás a TELJES teszthalmazon (kivettem a [::10]-et!)
run_evaluation_multi_k(test_playlists[::10], k_list=[10, 20, 50, 100, 500])

📂 Adatok betöltése...
🏗️ Modell felépítése és súlyok betöltése...
✅ Kaggle W2V súlyok sikeresen betöltve!
✂️ Inferencia modell kivágása a jósláshoz...
⚡ Finomhangolt W2V vektorok kinyerése a modellből...
🏗️ FAISS index építése...
📊 FAISS Indexben tárolt dalok száma: 372433

🧐 Fair kiértékelés indítása 9,918 listán...


Evaluating:   0%|          | 0/9918 [00:00<?, ?it/s]


🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥 EREDMÉNYEK (TISZTA W2V) 🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
--- Top 10 Ajánlás ---
✨ Hit Rate:  9.6189%
📈 NDCG:      5.3139%
🎯 MAP:       4.0121%

--- Top 20 Ajánlás ---
✨ Hit Rate:  13.5410%
📈 NDCG:      6.3025%
🎯 MAP:       4.2819%

--- Top 50 Ajánlás ---
✨ Hit Rate:  20.2964%
📈 NDCG:      7.6383%
🎯 MAP:       4.4944%

--- Top 100 Ajánlás ---
✨ Hit Rate:  26.9409%
📈 NDCG:      8.7148%
🎯 MAP:       4.5889%

--- Top 500 Ajánlás ---
✨ Hit Rate:  45.6140%
📈 NDCG:      11.1070%
🎯 MAP:       4.6782%

🖱️ Átlagos RecSys Clicks: 33.0852


# Bug fix

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import faiss
import os
import gc
from tqdm.auto import tqdm

# ==========================================
# 1. ÚTVONALAK (Állítsd be a saját gépedhez!)
# ==========================================
BASE_DIR = "../Models" 

HYBRID_MATRIX_PATH = os.path.join(BASE_DIR, "word2vec_matrix.npy")
TEST_DATA_PATH = os.path.join(BASE_DIR, "test_pids.npy")
WEIGHTS_PATH = os.path.join(BASE_DIR, "best_sasrec_large_hard_w2v_only.weights.h5")

# ==========================================
# 2. HIPERPARAMÉTEREK & ADATBETÖLTÉS
# ==========================================
MAX_LEN = 50

print("📂 Adatok betöltése...")
w2v_weights = np.load(HYBRID_MATRIX_PATH).astype('float32')
vocab_size = w2v_weights.shape[0]
D_MODEL = w2v_weights.shape[1]  # Dinamikus dimenzió!
test_playlists = np.load(TEST_DATA_PATH, allow_pickle=True)

# ÚJ: 0 indexű dalok eltávolítása minden listából
test_playlists = np.array([
    [t for t in pl if t != 0] for pl in test_playlists
], dtype=object)

# ==========================================
# 3. AZ ÚJ, NAGY ARCHITEKTÚRA
# ==========================================
def create_large_bpr_model(vocab_size, init_weights):
    seq_inputs = layers.Input(shape=(MAX_LEN,), name="seq_inputs")
    pos_inputs = layers.Input(shape=(MAX_LEN,), name="pos_inputs")
    neg_inputs = layers.Input(shape=(MAX_LEN,), name="neg_inputs")

    item_embedding = layers.Embedding(
        vocab_size, D_MODEL, mask_zero=True,
        embeddings_initializer=tf.keras.initializers.Constant(init_weights),
        trainable=True, name="w2v_embedding" # <--- JAVÍTVA!
    )
    pos_embedding = layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")

    x = item_embedding(seq_inputs)
    x += pos_embedding(tf.range(MAX_LEN)[tf.newaxis, :])
    
    for i in range(3): 
        attn = layers.MultiHeadAttention(num_heads=8, key_dim=D_MODEL//8, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))
        
    seq_states = layers.Dense(D_MODEL, name="final_projection")(x)

    pos_emb = item_embedding(pos_inputs)
    neg_emb = item_embedding(neg_inputs)

    def dot_and_cast(tensors):
        seq, emb = tensors
        scores = tf.reduce_sum(seq * emb, axis=-1, keepdims=True)
        return tf.cast(scores, tf.float32)

    dot_layer = layers.Lambda(dot_and_cast, name="dot_scoring")
    
    pos_scores = dot_layer([seq_states, pos_emb])
    neg_scores = dot_layer([seq_states, neg_emb])

    outputs = layers.Concatenate(axis=-1, name="concat_scores")([pos_scores, neg_scores])
    return tf.keras.Model(inputs=[seq_inputs, pos_inputs, neg_inputs], outputs=outputs)

print("🏗️ Modell felépítése és súlyok betöltése...")
model = create_large_bpr_model(vocab_size, w2v_weights)
model.load_weights(WEIGHTS_PATH)
print("✅ Kaggle W2V súlyok sikeresen betöltve!")

del w2v_weights
gc.collect()

# ==========================================
# 4. INFERENCIA & FAISS (JAVÍTVA)
# ==========================================
print("✂️ Inferencia modell kivágása a jósláshoz...")
inf_inputs = model.inputs[0] 
inf_outputs = model.get_layer("final_projection").output
inference_model = tf.keras.Model(inputs=inf_inputs, outputs=inf_outputs)

print("⚡ Finomhangolt W2V vektorok kinyerése a modellből...")
# <--- JAVÍTVA A RÉTEGNÉV!
updated_embeddings = model.get_layer("w2v_embedding").get_weights()[0] 

print("🏗️ FAISS index építése...")
faiss_embeddings = updated_embeddings.copy().astype('float32')
faiss.normalize_L2(faiss_embeddings)
index = faiss.IndexFlatIP(D_MODEL)
index.add(faiss_embeddings)
print(f"📊 FAISS Indexben tárolt dalok száma: {index.ntotal}")

# ==========================================
# 6. MULTI-K FAIR KIÉRTÉKELŐ (+ CLICKS)
# ==========================================
def run_evaluation_multi_k(playlists, k_list=[10, 50, 100]):
    max_k = max(k_list)
    hits = {k: 0 for k in k_list}
    ndcg = {k: 0 for k in k_list}
    map_score = {k: 0 for k in k_list}
    total_clicks = 0
    total = 0
    
    print(f"\n🧐 Fair kiértékelés indítása {len(playlists):,} listán...")
    
    for pl in tqdm(playlists, desc="Evaluating"):
        if len(pl) < 2: continue
        
        context_indices = pl[:-1] 
        target_id = pl[-1]
        
        input_seq = context_indices[-MAX_LEN:]
        pad_len = MAX_LEN - len(input_seq)
        input_padded = np.array([[0] * pad_len + input_seq])
        
        pred_vectors = inference_model.predict(input_padded, verbose=0)
        target_pred_vector = pred_vectors[0, -1, :].reshape(1, -1).astype('float32')
        
        faiss.normalize_L2(target_pred_vector)
        _, raw_top_indices = index.search(target_pred_vector, max_k + MAX_LEN)
        raw_top_indices = raw_top_indices[0]
        
        recommendations = []
        for idx in raw_top_indices:
            if idx == 0:  # ÚJ: padding token kizárása
                continue
            if idx not in context_indices:
                recommendations.append(idx)
            if len(recommendations) == max_k:
                break
        
        total += 1
        
        # --- METRIKÁK ÉS CLICKS SZÁMÍTÁSA ---
        if target_id in recommendations:
            rank = recommendations.index(target_id) + 1
            current_clicks = (rank - 1) // 10  # RecSys Clicks matek
            
            for k in k_list:
                if rank <= k:
                    hits[k] += 1
                    ndcg[k] += 1 / np.log2(rank + 1)
                    map_score[k] += 1 / rank
        else:
            current_clicks = 51 # RecSys maximális büntetés
            
        total_clicks += current_clicks

    print("\n" + "🔥"*15 + " EREDMÉNYEK (TISZTA W2V) " + "🔥"*15)
    for k in k_list:
        print(f"--- Top {k} Ajánlás ---")
        print(f"✨ Hit Rate:  {hits[k]/total:.4%}")
        print(f"📈 NDCG:      {ndcg[k]/total:.4%}")
        print(f"🎯 MAP:       {map_score[k]/total:.4%}\n")
    print("="*45)
    # Az átlagos kattintásszám:
    print(f"🖱️ Átlagos RecSys Clicks: {total_clicks/total:.4f}")
    print("="*45)

# Indítás a TELJES teszthalmazon (kivettem a [::10]-et!)
run_evaluation_multi_k(test_playlists[::10], k_list=[10, 20, 50, 100, 500])

📂 Adatok betöltése...
🏗️ Modell felépítése és súlyok betöltése...


Evaluating:   0%|          | 0/9918 [00:00<?, ?it/s]


🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥 EREDMÉNYEK 🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
--- Top 10 ---
✨ Hit Rate:  9.6390%
📈 NDCG:      5.3418%
🎯 MAP:       4.0381%

--- Top 20 ---
✨ Hit Rate:  13.6318%
📈 NDCG:      6.3470%
🎯 MAP:       4.3117%

--- Top 50 ---
✨ Hit Rate:  20.2763%
📈 NDCG:      7.6544%
🎯 MAP:       4.5169%

--- Top 100 ---
✨ Hit Rate:  26.6888%
📈 NDCG:      8.6923%
🎯 MAP:       4.6077%

--- Top 500 ---
✨ Hit Rate:  45.6040%
📈 NDCG:      11.1192%
🎯 MAP:       4.6992%

🖱️ Átlagos RecSys Clicks: 33.1208

📊 DIAGNOSZTIKA: Top-500-ból kiesett targetek
Darabszám: 3077
Átlagos rang:      1906.5
Medián rang:       1559.0
90. percentilis:   3858.0
95. percentilis:   4350.2
Maximum rang:      5039.0


# Kiértékelés: W2V téren tanítás (trainable=True) és hibrid téren kiértékelés

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import faiss
import os
import gc
from tqdm.auto import tqdm

# ==========================================
# 1. ÚTVONALAK (Állítsd be a saját gépedhez!)
# ==========================================
BASE_DIR = "../Models" 

HYBRID_MATRIX_PATH = os.path.join(BASE_DIR, "hybrid_embedding_matrix.npy")
TEST_DATA_PATH = os.path.join(BASE_DIR, "test_pids.npy")
WEIGHTS_PATH = os.path.join(BASE_DIR, "best_sasrec_large_hard_w2v_only.weights.h5")

# ==========================================
# 2. HIPERPARAMÉTEREK & ADATBETÖLTÉS
# ==========================================
MAX_LEN = 50

print("📂 Adatok betöltése...")
w2v_weights = np.load(HYBRID_MATRIX_PATH).astype('float32')
vocab_size = w2v_weights.shape[0]
D_MODEL = w2v_weights.shape[1]  # Dinamikus dimenzió!
test_playlists = np.load(TEST_DATA_PATH, allow_pickle=True)

# ==========================================
# 3. AZ ÚJ, NAGY ARCHITEKTÚRA
# ==========================================
def create_large_bpr_model(vocab_size, init_weights):
    seq_inputs = layers.Input(shape=(MAX_LEN,), name="seq_inputs")
    pos_inputs = layers.Input(shape=(MAX_LEN,), name="pos_inputs")
    neg_inputs = layers.Input(shape=(MAX_LEN,), name="neg_inputs")

    item_embedding = layers.Embedding(
        vocab_size, D_MODEL, mask_zero=True,
        embeddings_initializer=tf.keras.initializers.Constant(init_weights),
        trainable=True, name="w2v_embedding" # <--- JAVÍTVA!
    )
    pos_embedding = layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")

    x = item_embedding(seq_inputs)
    x += pos_embedding(tf.range(MAX_LEN)[tf.newaxis, :])
    
    for i in range(3): 
        attn = layers.MultiHeadAttention(num_heads=8, key_dim=D_MODEL//8, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))
        
    seq_states = layers.Dense(D_MODEL, name="final_projection")(x)

    pos_emb = item_embedding(pos_inputs)
    neg_emb = item_embedding(neg_inputs)

    def dot_and_cast(tensors):
        seq, emb = tensors
        scores = tf.reduce_sum(seq * emb, axis=-1, keepdims=True)
        return tf.cast(scores, tf.float32)

    dot_layer = layers.Lambda(dot_and_cast, name="dot_scoring")
    
    pos_scores = dot_layer([seq_states, pos_emb])
    neg_scores = dot_layer([seq_states, neg_emb])

    outputs = layers.Concatenate(axis=-1, name="concat_scores")([pos_scores, neg_scores])
    return tf.keras.Model(inputs=[seq_inputs, pos_inputs, neg_inputs], outputs=outputs)

print("🏗️ Modell felépítése és súlyok betöltése...")
model = create_large_bpr_model(vocab_size, w2v_weights)
model.load_weights(WEIGHTS_PATH)
print("✅ Kaggle W2V súlyok sikeresen betöltve!")

del w2v_weights
gc.collect()

# ==========================================
# 4. INFERENCIA & FAISS (JAVÍTVA)
# ==========================================
print("✂️ Inferencia modell kivágása a jósláshoz...")
inf_inputs = model.inputs[0] 
inf_outputs = model.get_layer("final_projection").output
inference_model = tf.keras.Model(inputs=inf_inputs, outputs=inf_outputs)

print("⚡ Finomhangolt W2V vektorok kinyerése a modellből...")
# <--- JAVÍTVA A RÉTEGNÉV!
updated_embeddings = model.get_layer("w2v_embedding").get_weights()[0] 

print("🏗️ FAISS index építése...")
faiss_embeddings = updated_embeddings.copy().astype('float32')
faiss.normalize_L2(faiss_embeddings)
index = faiss.IndexFlatIP(D_MODEL)
index.add(faiss_embeddings)
print(f"📊 FAISS Indexben tárolt dalok száma: {index.ntotal}")

# ==========================================
# 6. MULTI-K FAIR KIÉRTÉKELŐ (+ CLICKS)
# ==========================================
def run_evaluation_multi_k(playlists, k_list=[10, 50, 100]):
    max_k = max(k_list)
    hits = {k: 0 for k in k_list}
    ndcg = {k: 0 for k in k_list}
    map_score = {k: 0 for k in k_list}
    total_clicks = 0
    total = 0
    
    print(f"\n🧐 Fair kiértékelés indítása {len(playlists):,} listán...")
    
    for pl in tqdm(playlists, desc="Evaluating"):
        if len(pl) < 2: continue
        
        context_indices = pl[:-1] 
        target_id = pl[-1]
        
        input_seq = context_indices[-MAX_LEN:]
        pad_len = MAX_LEN - len(input_seq)
        input_padded = np.array([[0] * pad_len + input_seq])
        
        pred_vectors = inference_model.predict(input_padded, verbose=0)
        target_pred_vector = pred_vectors[0, -1, :].reshape(1, -1).astype('float32')
        
        faiss.normalize_L2(target_pred_vector)
        _, raw_top_indices = index.search(target_pred_vector, max_k + len(context_indices))
        raw_top_indices = raw_top_indices[0]
        
        recommendations = []
        for idx in raw_top_indices:
            if idx not in context_indices:
                recommendations.append(idx)
            if len(recommendations) == max_k:
                break
        
        total += 1
        
        # --- METRIKÁK ÉS CLICKS SZÁMÍTÁSA ---
        if target_id in recommendations:
            rank = recommendations.index(target_id) + 1
            current_clicks = (rank - 1) // 10  # RecSys Clicks matek
            
            for k in k_list:
                if rank <= k:
                    hits[k] += 1
                    ndcg[k] += 1 / np.log2(rank + 1)
                    map_score[k] += 1 / rank
        else:
            current_clicks = 51 # RecSys maximális büntetés
            
        total_clicks += current_clicks

    print("\n" + "🔥"*15 + " EREDMÉNYEK (TISZTA W2V) " + "🔥"*15)
    for k in k_list:
        print(f"--- Top {k} Ajánlás ---")
        print(f"✨ Hit Rate:  {hits[k]/total:.4%}")
        print(f"📈 NDCG:      {ndcg[k]/total:.4%}")
        print(f"🎯 MAP:       {map_score[k]/total:.4%}\n")
    print("="*45)
    # Az átlagos kattintásszám:
    print(f"🖱️ Átlagos RecSys Clicks: {total_clicks/total:.4f}")
    print("="*45)

# Indítás a TELJES teszthalmazon (kivettem a [::10]-et!)
run_evaluation_multi_k(test_playlists[::10], k_list=[10, 20, 50, 100, 500])

📂 Adatok betöltése...
🏗️ Modell felépítése és súlyok betöltése...



c:\Users\Béres Gábor\music_recommender\.venv\Lib\site-packages\keras\src\layers\layer.py:982: UserWarning: Layer 'dot_scoring' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


✅ Kaggle W2V súlyok sikeresen betöltve!
✂️ Inferencia modell kivágása a jósláshoz...
⚡ Finomhangolt W2V vektorok kinyerése a modellből...
🏗️ FAISS index építése...
📊 FAISS Indexben tárolt dalok száma: 372433

🧐 Fair kiértékelés indítása 9,918 listán...


Evaluating:   0%|          | 0/9918 [00:00<?, ?it/s]


🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥 EREDMÉNYEK (TISZTA W2V) 🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
--- Top 10 Ajánlás ---
✨ Hit Rate:  9.7399%
📈 NDCG:      5.4194%
🎯 MAP:       4.1075%

--- Top 20 Ajánlás ---
✨ Hit Rate:  13.7629%
📈 NDCG:      6.4353%
🎯 MAP:       4.3857%

--- Top 50 Ajánlás ---
✨ Hit Rate:  20.5384%
📈 NDCG:      7.7750%
🎯 MAP:       4.5987%

--- Top 100 Ajánlás ---
✨ Hit Rate:  27.2938%
📈 NDCG:      8.8700%
🎯 MAP:       4.6950%

--- Top 500 Ajánlás ---
✨ Hit Rate:  46.1081%
📈 NDCG:      11.2789%
🎯 MAP:       4.7848%

🖱️ Átlagos RecSys Clicks: 32.9046


# Több elem jósolása

In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import faiss
import os
import gc
from tqdm.auto import tqdm

# ==========================================
# 1. ÚTVONALAK (Állítsd be a saját gépedhez!)
# ==========================================
BASE_DIR = "../Models" 

HYBRID_MATRIX_PATH = os.path.join(BASE_DIR, "hybrid_embedding_matrix.npy")
TEST_DATA_PATH = os.path.join(BASE_DIR, "test_pids.npy")
WEIGHTS_PATH = os.path.join(BASE_DIR, "best_sasrec_large_hard_w2v_only.weights.h5")

# ==========================================
# 2. HIPERPARAMÉTEREK & ADATBETÖLTÉS
# ==========================================
MAX_LEN = 50

print("📂 Adatok betöltése...")
w2v_weights = np.load(HYBRID_MATRIX_PATH).astype('float32')
vocab_size = w2v_weights.shape[0]
D_MODEL = w2v_weights.shape[1]  # Dinamikus dimenzió!
test_playlists = np.load(TEST_DATA_PATH, allow_pickle=True)

# ==========================================
# 3. AZ ÚJ, NAGY ARCHITEKTÚRA
# ==========================================
def create_large_bpr_model(vocab_size, init_weights):
    seq_inputs = layers.Input(shape=(MAX_LEN,), name="seq_inputs")
    pos_inputs = layers.Input(shape=(MAX_LEN,), name="pos_inputs")
    neg_inputs = layers.Input(shape=(MAX_LEN,), name="neg_inputs")

    item_embedding = layers.Embedding(
        vocab_size, D_MODEL, mask_zero=True,
        embeddings_initializer=tf.keras.initializers.Constant(init_weights),
        trainable=True, name="w2v_embedding" # <--- JAVÍTVA!
    )
    pos_embedding = layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")

    x = item_embedding(seq_inputs)
    x += pos_embedding(tf.range(MAX_LEN)[tf.newaxis, :])
    
    for i in range(3): 
        attn = layers.MultiHeadAttention(num_heads=8, key_dim=D_MODEL//8, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))
        
    seq_states = layers.Dense(D_MODEL, name="final_projection")(x)

    pos_emb = item_embedding(pos_inputs)
    neg_emb = item_embedding(neg_inputs)

    def dot_and_cast(tensors):
        seq, emb = tensors
        scores = tf.reduce_sum(seq * emb, axis=-1, keepdims=True)
        return tf.cast(scores, tf.float32)

    dot_layer = layers.Lambda(dot_and_cast, name="dot_scoring")
    
    pos_scores = dot_layer([seq_states, pos_emb])
    neg_scores = dot_layer([seq_states, neg_emb])

    outputs = layers.Concatenate(axis=-1, name="concat_scores")([pos_scores, neg_scores])
    return tf.keras.Model(inputs=[seq_inputs, pos_inputs, neg_inputs], outputs=outputs)

print("🏗️ Modell felépítése és súlyok betöltése...")
model = create_large_bpr_model(vocab_size, w2v_weights)
model.load_weights(WEIGHTS_PATH)
print("✅ Kaggle W2V súlyok sikeresen betöltve!")

del w2v_weights
gc.collect()

# ==========================================
# 4. INFERENCIA & FAISS (JAVÍTVA)
# ==========================================
print("✂️ Inferencia modell kivágása a jósláshoz...")
inf_inputs = model.inputs[0] 
inf_outputs = model.get_layer("final_projection").output
inference_model = tf.keras.Model(inputs=inf_inputs, outputs=inf_outputs)

print("⚡ Finomhangolt W2V vektorok kinyerése a modellből...")
# <--- JAVÍTVA A RÉTEGNÉV!
updated_embeddings = model.get_layer("w2v_embedding").get_weights()[0] 

print("🏗️ FAISS index építése...")
faiss_embeddings = updated_embeddings.copy().astype('float32')
faiss.normalize_L2(faiss_embeddings)
index = faiss.IndexFlatIP(D_MODEL)
index.add(faiss_embeddings)
print(f"📊 FAISS Indexben tárolt dalok száma: {index.ntotal}")

# ==========================================
# 6. RECSYS CHALLENGE KIÉRTÉKELÉS (MULTI-TARGET)
# ==========================================
def run_evaluation_challenge_style(playlists, given_n=25, k_list=[10, 50, 100, 500]):
    max_k = max(k_list)
    
    # Eredménygyűjtők
    recall = {k: 0.0 for k in k_list}
    ndcg = {k: 0.0 for k in k_list}
    r_precision_total = 0.0
    total_clicks = 0
    total_valid_playlists = 0
    
    print(f"\n🧐 Challenge kiértékelés indítása (Kontextus: első {given_n} dal)...")
    
    for pl in tqdm(playlists, desc="Evaluating"):
        # Csak azokat a listákat nézzük, amik hosszabbak, mint a megadott seed (pl. 25)
        if len(pl) <= given_n: 
            continue
        
        # 1. SZÉTVÁGÁS: Kontextus és Célpontok
        context_indices = pl[:given_n] 
        target_ids = set(pl[given_n:]) # Set-et használunk a gyors kereséshez
        num_targets = len(target_ids)
        
        # 2. PREDIKCIÓ (Ugyanúgy, mint eddig)
        input_seq = context_indices[-MAX_LEN:]
        pad_len = MAX_LEN - len(input_seq)
        input_padded = np.array([[0] * pad_len + input_seq])
        
        pred_vectors = inference_model.predict(input_padded, verbose=0)
        target_pred_vector = pred_vectors[0, -1, :].reshape(1, -1).astype('float32')
        
        faiss.normalize_L2(target_pred_vector)
        # Kérünk max_k + kontextus méretű jelöltet, hogy a már hallgatottakat ki tudjuk szűrni
        _, raw_top_indices = index.search(target_pred_vector, max_k + len(context_indices))
        raw_top_indices = raw_top_indices[0]
        
        # 3. SZŰRÉS (Már hallgatott dalok kivétele)
        recommendations = []
        for idx in raw_top_indices:
            if idx not in context_indices:
                recommendations.append(idx)
            if len(recommendations) == max_k:
                break
                
        total_valid_playlists += 1
        
        # 4. TALÁLATOK RANKSZÁMÁNAK KIGYŰJTÉSE
        hits_ranks = []
        for i, rec_id in enumerate(recommendations):
            if rec_id in target_ids:
                hits_ranks.append(i + 1) # 1-alapú sorszám
                
        # ==========================================
        # METRIKÁK SZÁMÍTÁSA (MULTI-TARGET KÉPLETEK)
        # ==========================================
        
        # A) CLICKS (Hány lapozás az ELSŐ találatig?)
        if hits_ranks:
            first_hit_rank = hits_ranks[0]
            current_clicks = (first_hit_rank - 1) // 10
        else:
            current_clicks = 51 # Max büntetés
        total_clicks += current_clicks
        
        # B) R-PRECISION (RecSys hivatalos metrika)
        # Hány találat van az első R ajánlásban? (Ahol R = a hiányzó dalok száma)
        R = num_targets
        hits_in_R = sum(1 for r in hits_ranks if r <= R)
        r_precision_total += (hits_in_R / R)
        
        # C) RECALL@K és NDCG@K
        for k in k_list:
            # Csak a top K-ba eső találatok
            k_hits_ranks = [r for r in hits_ranks if r <= k]
            
            # Recall: A hiányzó dalok hány százalékát találtuk meg a top K-ban?
            recall[k] += len(k_hits_ranks) / num_targets
            
            # NDCG kiszámítása (DCG / IDCG)
            dcg = sum(1.0 / np.log2(rank + 1) for rank in k_hits_ranks)
            
            # IDCG: Mi lenne a tökéletes elrendezés? (Minden jó dal a leggelején van)
            idcg_len = min(k, num_targets)
            idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, idcg_len + 1))
            
            if idcg > 0:
                ndcg[k] += dcg / idcg

    # ==========================================
    # EREDMÉNYEK KIÍRÁSA
    # ==========================================
    print("\n" + "🔥"*15 + f" EREDMÉNYEK (First {given_n} songs) " + "🔥"*15)
    print(f"Értékelt listák száma: {total_valid_playlists:,}\n")
    
    # R-Precision és Clicks globális átlaga
    print(f"🎯 R-Precision: {r_precision_total / total_valid_playlists:.4%}")
    print(f"🖱️ Clicks:      {total_clicks / total_valid_playlists:.4f}\n")
    print("-" * 50)
    
    for k in k_list:
        print(f"--- Top {k} Ajánlás ---")
        print(f"✨ Recall@{k}: {recall[k]/total_valid_playlists:.4%}")
        print(f"📈 NDCG@{k}:   {ndcg[k]/total_valid_playlists:.4%}\n")
    print("="*60)

# Indítás a teszthalmazon: Az első 25 dal alapján jósoljuk a többit
run_evaluation_challenge_style(test_playlists[::10], given_n=25, k_list=[10, 50, 100, 500])

📂 Adatok betöltése...
🏗️ Modell felépítése és súlyok betöltése...
✅ Kaggle W2V súlyok sikeresen betöltve!
✂️ Inferencia modell kivágása a jósláshoz...
⚡ Finomhangolt W2V vektorok kinyerése a modellből...
🏗️ FAISS index építése...
📊 FAISS Indexben tárolt dalok száma: 372433

🧐 Challenge kiértékelés indítása (Kontextus: első 25 dal)...


Evaluating:   0%|          | 0/9918 [00:00<?, ?it/s]


🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥 EREDMÉNYEK (First 25 songs) 🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
Értékelt listák száma: 7,274

🎯 R-Precision: 10.1682%
🖱️ Clicks:      4.4993

--------------------------------------------------
--- Top 10 Ajánlás ---
✨ Recall@10: 4.6529%
📈 NDCG@10:   16.9866%

--- Top 50 Ajánlás ---
✨ Recall@50: 13.0820%
📈 NDCG@50:   15.1086%

--- Top 100 Ajánlás ---
✨ Recall@100: 19.1810%
📈 NDCG@100:   16.2976%

--- Top 500 Ajánlás ---
✨ Recall@500: 38.4153%
📈 NDCG@500:   24.0145%



In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import faiss
import os
import gc
from tqdm.auto import tqdm

# ==========================================
# 1. ÚTVONALAK (Állítsd be a saját gépedhez!)
# ==========================================
BASE_DIR = "../Models" 

HYBRID_MATRIX_PATH = os.path.join(BASE_DIR, "hybrid_embedding_matrix.npy")
TEST_DATA_PATH = os.path.join(BASE_DIR, "test_pids.npy")
WEIGHTS_PATH = os.path.join(BASE_DIR, "best_sasrec_large_hard_w2v_only.weights.h5")

# ==========================================
# 2. HIPERPARAMÉTEREK & ADATBETÖLTÉS
# ==========================================
MAX_LEN = 50

print("📂 Adatok betöltése...")
w2v_weights = np.load(HYBRID_MATRIX_PATH).astype('float32')
vocab_size = w2v_weights.shape[0]
D_MODEL = w2v_weights.shape[1]  # Dinamikus dimenzió!
test_playlists = np.load(TEST_DATA_PATH, allow_pickle=True)

# ==========================================
# 3. AZ ÚJ, NAGY ARCHITEKTÚRA
# ==========================================
def create_large_bpr_model(vocab_size, init_weights):
    seq_inputs = layers.Input(shape=(MAX_LEN,), name="seq_inputs")
    pos_inputs = layers.Input(shape=(MAX_LEN,), name="pos_inputs")
    neg_inputs = layers.Input(shape=(MAX_LEN,), name="neg_inputs")

    item_embedding = layers.Embedding(
        vocab_size, D_MODEL, mask_zero=True,
        embeddings_initializer=tf.keras.initializers.Constant(init_weights),
        trainable=True, name="w2v_embedding" # <--- JAVÍTVA!
    )
    pos_embedding = layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")

    x = item_embedding(seq_inputs)
    x += pos_embedding(tf.range(MAX_LEN)[tf.newaxis, :])
    
    for i in range(3): 
        attn = layers.MultiHeadAttention(num_heads=8, key_dim=D_MODEL//8, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))
        
    seq_states = layers.Dense(D_MODEL, name="final_projection")(x)

    pos_emb = item_embedding(pos_inputs)
    neg_emb = item_embedding(neg_inputs)

    def dot_and_cast(tensors):
        seq, emb = tensors
        scores = tf.reduce_sum(seq * emb, axis=-1, keepdims=True)
        return tf.cast(scores, tf.float32)

    dot_layer = layers.Lambda(dot_and_cast, name="dot_scoring")
    
    pos_scores = dot_layer([seq_states, pos_emb])
    neg_scores = dot_layer([seq_states, neg_emb])

    outputs = layers.Concatenate(axis=-1, name="concat_scores")([pos_scores, neg_scores])
    return tf.keras.Model(inputs=[seq_inputs, pos_inputs, neg_inputs], outputs=outputs)

print("🏗️ Modell felépítése és súlyok betöltése...")
model = create_large_bpr_model(vocab_size, w2v_weights)
model.load_weights(WEIGHTS_PATH)
print("✅ Kaggle W2V súlyok sikeresen betöltve!")

del w2v_weights
gc.collect()

# ==========================================
# 4. INFERENCIA & FAISS (JAVÍTVA)
# ==========================================
print("✂️ Inferencia modell kivágása a jósláshoz...")
inf_inputs = model.inputs[0] 
inf_outputs = model.get_layer("final_projection").output
inference_model = tf.keras.Model(inputs=inf_inputs, outputs=inf_outputs)

print("⚡ Finomhangolt W2V vektorok kinyerése a modellből...")
# <--- JAVÍTVA A RÉTEGNÉV!
updated_embeddings = model.get_layer("w2v_embedding").get_weights()[0] 

print("🏗️ FAISS index építése...")
faiss_embeddings = updated_embeddings.copy().astype('float32')
faiss.normalize_L2(faiss_embeddings)
index = faiss.IndexFlatIP(D_MODEL)
index.add(faiss_embeddings)
print(f"📊 FAISS Indexben tárolt dalok száma: {index.ntotal}")

# ==========================================
# 6. RECSYS CHALLENGE KIÉRTÉKELÉS (MULTI-TARGET)
# ==========================================
def run_evaluation_challenge_style(playlists, given_n=25, k_list=[10, 50, 100, 500]):
    max_k = max(k_list)
    
    # Eredménygyűjtők
    recall = {k: 0.0 for k in k_list}
    ndcg = {k: 0.0 for k in k_list}
    r_precision_total = 0.0
    total_clicks = 0
    total_valid_playlists = 0
    
    print(f"\n🧐 Challenge kiértékelés indítása (Kontextus: első {given_n} dal)...")
    
    for pl in tqdm(playlists, desc="Evaluating"):
        # Csak azokat a listákat nézzük, amik hosszabbak, mint a megadott seed (pl. 25)
        if len(pl) <= given_n: 
            continue
        
        # 1. SZÉTVÁGÁS: Kontextus és Célpontok
        context_indices = pl[:given_n] 
        target_ids = set(pl[given_n:]) # Set-et használunk a gyors kereséshez
        num_targets = len(target_ids)
        
        # 2. PREDIKCIÓ (Ugyanúgy, mint eddig)
        input_seq = context_indices[-MAX_LEN:]
        pad_len = MAX_LEN - len(input_seq)
        input_padded = np.array([[0] * pad_len + input_seq])
        
        pred_vectors = inference_model.predict(input_padded, verbose=0)
        target_pred_vector = pred_vectors[0, -1, :].reshape(1, -1).astype('float32')
        
        faiss.normalize_L2(target_pred_vector)
        # Kérünk max_k + kontextus méretű jelöltet, hogy a már hallgatottakat ki tudjuk szűrni
        _, raw_top_indices = index.search(target_pred_vector, max_k + len(context_indices))
        raw_top_indices = raw_top_indices[0]
        
        # 3. SZŰRÉS (Már hallgatott dalok kivétele)
        recommendations = []
        for idx in raw_top_indices:
            if idx not in context_indices:
                recommendations.append(idx)
            if len(recommendations) == max_k:
                break
                
        total_valid_playlists += 1
        
        # 4. TALÁLATOK RANKSZÁMÁNAK KIGYŰJTÉSE
        hits_ranks = []
        for i, rec_id in enumerate(recommendations):
            if rec_id in target_ids:
                hits_ranks.append(i + 1) # 1-alapú sorszám
                
        # ==========================================
        # METRIKÁK SZÁMÍTÁSA (MULTI-TARGET KÉPLETEK)
        # ==========================================
        
        # A) CLICKS (Hány lapozás az ELSŐ találatig?)
        if hits_ranks:
            first_hit_rank = hits_ranks[0]
            current_clicks = (first_hit_rank - 1) // 10
        else:
            current_clicks = 51 # Max büntetés
        total_clicks += current_clicks
        
        # B) R-PRECISION (RecSys hivatalos metrika)
        # Hány találat van az első R ajánlásban? (Ahol R = a hiányzó dalok száma)
        R = num_targets
        hits_in_R = sum(1 for r in hits_ranks if r <= R)
        r_precision_total += (hits_in_R / R)
        
        # C) RECALL@K és NDCG@K
        for k in k_list:
            # Csak a top K-ba eső találatok
            k_hits_ranks = [r for r in hits_ranks if r <= k]
            
            # Recall: A hiányzó dalok hány százalékát találtuk meg a top K-ban?
            recall[k] += len(k_hits_ranks) / num_targets
            
            # NDCG kiszámítása (DCG / IDCG)
            dcg = sum(1.0 / np.log2(rank + 1) for rank in k_hits_ranks)
            
            # IDCG: Mi lenne a tökéletes elrendezés? (Minden jó dal a leggelején van)
            idcg_len = min(k, num_targets)
            idcg = sum(1.0 / np.log2(rank + 1) for rank in range(1, idcg_len + 1))
            
            if idcg > 0:
                ndcg[k] += dcg / idcg

    # ==========================================
    # EREDMÉNYEK KIÍRÁSA
    # ==========================================
    print("\n" + "🔥"*15 + f" EREDMÉNYEK (First {given_n} songs) " + "🔥"*15)
    print(f"Értékelt listák száma: {total_valid_playlists:,}\n")
    
    # R-Precision és Clicks globális átlaga
    print(f"🎯 R-Precision: {r_precision_total / total_valid_playlists:.4%}")
    print(f"🖱️ Clicks:      {total_clicks / total_valid_playlists:.4f}\n")
    print("-" * 50)
    
    for k in k_list:
        print(f"--- Top {k} Ajánlás ---")
        print(f"✨ Recall@{k}: {recall[k]/total_valid_playlists:.4%}")
        print(f"📈 NDCG@{k}:   {ndcg[k]/total_valid_playlists:.4%}\n")
    print("="*60)

# Indítás a teszthalmazon: Az első 25 dal alapján jósoljuk a többit
run_evaluation_challenge_style(test_playlists[::100], given_n=10, k_list=[10, 50, 100, 500])

📂 Adatok betöltése...
🏗️ Modell felépítése és súlyok betöltése...
✅ Kaggle W2V súlyok sikeresen betöltve!
✂️ Inferencia modell kivágása a jósláshoz...
⚡ Finomhangolt W2V vektorok kinyerése a modellből...
🏗️ FAISS index építése...
📊 FAISS Indexben tárolt dalok száma: 372433

🧐 Challenge kiértékelés indítása (Kontextus: első 10 dal)...


Evaluating:   0%|          | 0/992 [00:00<?, ?it/s]


🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥 EREDMÉNYEK (First 10 songs) 🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
Értékelt listák száma: 928

🎯 R-Precision: 11.3142%
🖱️ Clicks:      4.1994

--------------------------------------------------
--- Top 10 Ajánlás ---
✨ Recall@10: 5.4364%
📈 NDCG@10:   19.2929%

--- Top 50 Ajánlás ---
✨ Recall@50: 14.5354%
📈 NDCG@50:   16.8178%

--- Top 100 Ajánlás ---
✨ Recall@100: 21.0475%
📈 NDCG@100:   18.1927%

--- Top 500 Ajánlás ---
✨ Recall@500: 39.7568%
📈 NDCG@500:   25.6123%



## Más tér

In [7]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import faiss
import os
import gc
from tqdm.auto import tqdm

# ==========================================
# 1. ÚTVONALAK (Állítsd be a saját gépedhez!)
# ==========================================
BASE_DIR = "../Models" 

HYBRID_MATRIX_PATH = os.path.join(BASE_DIR, "hybrid_embedding_matrix_mel.npy")
TEST_DATA_PATH = os.path.join(BASE_DIR, "test_pids.npy")
WEIGHTS_PATH = os.path.join(BASE_DIR, "best_sasrec_large_hard_w2v_only.weights.h5")

# ==========================================
# 2. HIPERPARAMÉTEREK & ADATBETÖLTÉS
# ==========================================
MAX_LEN = 50

print("📂 Adatok betöltése...")
w2v_weights = np.load(HYBRID_MATRIX_PATH).astype('float32')
vocab_size = w2v_weights.shape[0]
D_MODEL = w2v_weights.shape[1]  # Dinamikus dimenzió!
test_playlists = np.load(TEST_DATA_PATH, allow_pickle=True)

# ==========================================
# 3. AZ ÚJ, NAGY ARCHITEKTÚRA
# ==========================================
def create_large_bpr_model(vocab_size, init_weights):
    seq_inputs = layers.Input(shape=(MAX_LEN,), name="seq_inputs")
    pos_inputs = layers.Input(shape=(MAX_LEN,), name="pos_inputs")
    neg_inputs = layers.Input(shape=(MAX_LEN,), name="neg_inputs")

    item_embedding = layers.Embedding(
        vocab_size, D_MODEL, mask_zero=True,
        embeddings_initializer=tf.keras.initializers.Constant(init_weights),
        trainable=True, name="w2v_embedding" # <--- JAVÍTVA!
    )
    pos_embedding = layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")

    x = item_embedding(seq_inputs)
    x += pos_embedding(tf.range(MAX_LEN)[tf.newaxis, :])
    
    for i in range(3): 
        attn = layers.MultiHeadAttention(num_heads=8, key_dim=D_MODEL//8, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))
        
    seq_states = layers.Dense(D_MODEL, name="final_projection")(x)

    pos_emb = item_embedding(pos_inputs)
    neg_emb = item_embedding(neg_inputs)

    def dot_and_cast(tensors):
        seq, emb = tensors
        scores = tf.reduce_sum(seq * emb, axis=-1, keepdims=True)
        return tf.cast(scores, tf.float32)

    dot_layer = layers.Lambda(dot_and_cast, name="dot_scoring")
    
    pos_scores = dot_layer([seq_states, pos_emb])
    neg_scores = dot_layer([seq_states, neg_emb])

    outputs = layers.Concatenate(axis=-1, name="concat_scores")([pos_scores, neg_scores])
    return tf.keras.Model(inputs=[seq_inputs, pos_inputs, neg_inputs], outputs=outputs)

print("🏗️ Modell felépítése és súlyok betöltése...")
model = create_large_bpr_model(vocab_size, w2v_weights)
model.load_weights(WEIGHTS_PATH)
print("✅ Kaggle W2V súlyok sikeresen betöltve!")

del w2v_weights
gc.collect()

# ==========================================
# 4. INFERENCIA & FAISS (JAVÍTVA)
# ==========================================
print("✂️ Inferencia modell kivágása a jósláshoz...")
inf_inputs = model.inputs[0] 
inf_outputs = model.get_layer("final_projection").output
inference_model = tf.keras.Model(inputs=inf_inputs, outputs=inf_outputs)

print("⚡ Finomhangolt W2V vektorok kinyerése a modellből...")
# <--- JAVÍTVA A RÉTEGNÉV!
updated_embeddings = model.get_layer("w2v_embedding").get_weights()[0] 

print("🏗️ FAISS index építése...")
faiss_embeddings = updated_embeddings.copy().astype('float32')
faiss.normalize_L2(faiss_embeddings)
index = faiss.IndexFlatIP(D_MODEL)
index.add(faiss_embeddings)
print(f"📊 FAISS Indexben tárolt dalok száma: {index.ntotal}")

# ==========================================
# 6. MULTI-K FAIR KIÉRTÉKELŐ (+ CLICKS)
# ==========================================
def run_evaluation_multi_k(playlists, k_list=[10, 50, 100]):
    max_k = max(k_list)
    hits = {k: 0 for k in k_list}
    ndcg = {k: 0 for k in k_list}
    map_score = {k: 0 for k in k_list}
    total_clicks = 0
    total = 0
    
    print(f"\n🧐 Fair kiértékelés indítása {len(playlists):,} listán...")
    
    for pl in tqdm(playlists, desc="Evaluating"):
        if len(pl) < 2: continue
        
        context_indices = pl[:-1] 
        target_id = pl[-1]
        
        input_seq = context_indices[-MAX_LEN:]
        pad_len = MAX_LEN - len(input_seq)
        input_padded = np.array([[0] * pad_len + input_seq])
        
        pred_vectors = inference_model.predict(input_padded, verbose=0)
        target_pred_vector = pred_vectors[0, -1, :].reshape(1, -1).astype('float32')
        
        faiss.normalize_L2(target_pred_vector)
        _, raw_top_indices = index.search(target_pred_vector, max_k + len(context_indices))
        raw_top_indices = raw_top_indices[0]
        
        recommendations = []
        for idx in raw_top_indices:
            if idx not in context_indices:
                recommendations.append(idx)
            if len(recommendations) == max_k:
                break
        
        total += 1
        
        # --- METRIKÁK ÉS CLICKS SZÁMÍTÁSA ---
        if target_id in recommendations:
            rank = recommendations.index(target_id) + 1
            current_clicks = (rank - 1) // 10  # RecSys Clicks matek
            
            for k in k_list:
                if rank <= k:
                    hits[k] += 1
                    ndcg[k] += 1 / np.log2(rank + 1)
                    map_score[k] += 1 / rank
        else:
            current_clicks = 51 # RecSys maximális büntetés
            
        total_clicks += current_clicks

    print("\n" + "🔥"*15 + " EREDMÉNYEK (TISZTA W2V) " + "🔥"*15)
    for k in k_list:
        print(f"--- Top {k} Ajánlás ---")
        print(f"✨ Hit Rate:  {hits[k]/total:.4%}")
        print(f"📈 NDCG:      {ndcg[k]/total:.4%}")
        print(f"🎯 MAP:       {map_score[k]/total:.4%}\n")
    print("="*45)
    # Az átlagos kattintásszám:
    print(f"🖱️ Átlagos RecSys Clicks: {total_clicks/total:.4f}")
    print("="*45)

# Indítás a TELJES teszthalmazon (kivettem a [::10]-et!)
run_evaluation_multi_k(test_playlists[::10], k_list=[10, 20, 50, 100, 500])

📂 Adatok betöltése...
🏗️ Modell felépítése és súlyok betöltése...
✅ Kaggle W2V súlyok sikeresen betöltve!
✂️ Inferencia modell kivágása a jósláshoz...
⚡ Finomhangolt W2V vektorok kinyerése a modellből...
🏗️ FAISS index építése...
📊 FAISS Indexben tárolt dalok száma: 372433

🧐 Fair kiértékelés indítása 9,918 listán...


Evaluating:   0%|          | 0/9918 [00:00<?, ?it/s]


🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥 EREDMÉNYEK (TISZTA W2V) 🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
--- Top 10 Ajánlás ---
✨ Hit Rate:  9.6592%
📈 NDCG:      5.3068%
🎯 MAP:       3.9844%

--- Top 20 Ajánlás ---
✨ Hit Rate:  13.7528%
📈 NDCG:      6.3380%
🎯 MAP:       4.2655%

--- Top 50 Ajánlás ---
✨ Hit Rate:  20.5384%
📈 NDCG:      7.6734%
🎯 MAP:       4.4754%

--- Top 100 Ajánlás ---
✨ Hit Rate:  27.0417%
📈 NDCG:      8.7272%
🎯 MAP:       4.5679%

--- Top 500 Ajánlás ---
✨ Hit Rate:  46.0173%
📈 NDCG:      11.1573%
🎯 MAP:       4.6586%

🖱️ Átlagos RecSys Clicks: 32.9720


# Frozen tér

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import faiss
import os
import gc
from tqdm.auto import tqdm

# ==========================================
# 1. ÚTVONALAK (Állítsd be a saját gépedhez!)
# ==========================================
BASE_DIR = "../Models" 

HYBRID_MATRIX_PATH = os.path.join(BASE_DIR, "hybrid_embedding_matrix.npy")
TEST_DATA_PATH = os.path.join(BASE_DIR, "test_pids.npy")
WEIGHTS_PATH = os.path.join(BASE_DIR, "best_sasrec_large_hard_w2v_frozen.weights.h5")

# ==========================================
# 2. HIPERPARAMÉTEREK & ADATBETÖLTÉS
# ==========================================
MAX_LEN = 50

print("📂 Adatok betöltése...")
w2v_weights = np.load(HYBRID_MATRIX_PATH).astype('float32')
vocab_size = w2v_weights.shape[0]
D_MODEL = w2v_weights.shape[1]  # Dinamikus dimenzió!
test_playlists = np.load(TEST_DATA_PATH, allow_pickle=True)

# ==========================================
# 3. AZ ÚJ, NAGY ARCHITEKTÚRA
# ==========================================
def create_large_bpr_model(vocab_size, init_weights):
    seq_inputs = layers.Input(shape=(MAX_LEN,), name="seq_inputs")
    pos_inputs = layers.Input(shape=(MAX_LEN,), name="pos_inputs")
    neg_inputs = layers.Input(shape=(MAX_LEN,), name="neg_inputs")

    item_embedding = layers.Embedding(
        vocab_size, D_MODEL, mask_zero=True,
        embeddings_initializer=tf.keras.initializers.Constant(init_weights),
        trainable=True, name="w2v_embedding" # <--- JAVÍTVA!
    )
    pos_embedding = layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")

    x = item_embedding(seq_inputs)
    x += pos_embedding(tf.range(MAX_LEN)[tf.newaxis, :])
    
    for i in range(3): 
        attn = layers.MultiHeadAttention(num_heads=8, key_dim=D_MODEL//8, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))
        
    seq_states = layers.Dense(D_MODEL, name="final_projection")(x)

    pos_emb = item_embedding(pos_inputs)
    neg_emb = item_embedding(neg_inputs)

    def dot_and_cast(tensors):
        seq, emb = tensors
        scores = tf.reduce_sum(seq * emb, axis=-1, keepdims=True)
        return tf.cast(scores, tf.float32)

    dot_layer = layers.Lambda(dot_and_cast, name="dot_scoring")
    
    pos_scores = dot_layer([seq_states, pos_emb])
    neg_scores = dot_layer([seq_states, neg_emb])

    outputs = layers.Concatenate(axis=-1, name="concat_scores")([pos_scores, neg_scores])
    return tf.keras.Model(inputs=[seq_inputs, pos_inputs, neg_inputs], outputs=outputs)

print("🏗️ Modell felépítése és súlyok betöltése...")
model = create_large_bpr_model(vocab_size, w2v_weights)
model.load_weights(WEIGHTS_PATH)
print("✅ Kaggle W2V súlyok sikeresen betöltve!")

del w2v_weights
gc.collect()

# ==========================================
# 4. INFERENCIA & FAISS (JAVÍTVA)
# ==========================================
print("✂️ Inferencia modell kivágása a jósláshoz...")
inf_inputs = model.inputs[0] 
inf_outputs = model.get_layer("final_projection").output
inference_model = tf.keras.Model(inputs=inf_inputs, outputs=inf_outputs)

print("⚡ Finomhangolt W2V vektorok kinyerése a modellből...")
# <--- JAVÍTVA A RÉTEGNÉV!
updated_embeddings = model.get_layer("w2v_embedding").get_weights()[0] 

print("🏗️ FAISS index építése...")
faiss_embeddings = updated_embeddings.copy().astype('float32')
faiss.normalize_L2(faiss_embeddings)
index = faiss.IndexFlatIP(D_MODEL)
index.add(faiss_embeddings)
print(f"📊 FAISS Indexben tárolt dalok száma: {index.ntotal}")

# ==========================================
# 6. MULTI-K FAIR KIÉRTÉKELŐ (+ CLICKS)
# ==========================================
def run_evaluation_multi_k(playlists, k_list=[10, 50, 100]):
    max_k = max(k_list)
    hits = {k: 0 for k in k_list}
    ndcg = {k: 0 for k in k_list}
    map_score = {k: 0 for k in k_list}
    total_clicks = 0
    total = 0
    
    print(f"\n🧐 Fair kiértékelés indítása {len(playlists):,} listán...")
    
    for pl in tqdm(playlists, desc="Evaluating"):
        if len(pl) < 2: continue
        
        context_indices = pl[:-1] 
        target_id = pl[-1]
        
        input_seq = context_indices[-MAX_LEN:]
        pad_len = MAX_LEN - len(input_seq)
        input_padded = np.array([[0] * pad_len + input_seq])
        
        pred_vectors = inference_model.predict(input_padded, verbose=0)
        target_pred_vector = pred_vectors[0, -1, :].reshape(1, -1).astype('float32')
        
        faiss.normalize_L2(target_pred_vector)
        _, raw_top_indices = index.search(target_pred_vector, max_k + len(context_indices))
        raw_top_indices = raw_top_indices[0]
        
        recommendations = []
        for idx in raw_top_indices:
            if idx not in context_indices:
                recommendations.append(idx)
            if len(recommendations) == max_k:
                break
        
        total += 1
        
        # --- METRIKÁK ÉS CLICKS SZÁMÍTÁSA ---
        if target_id in recommendations:
            rank = recommendations.index(target_id) + 1
            current_clicks = (rank - 1) // 10  # RecSys Clicks matek
            
            for k in k_list:
                if rank <= k:
                    hits[k] += 1
                    ndcg[k] += 1 / np.log2(rank + 1)
                    map_score[k] += 1 / rank
        else:
            current_clicks = 51 # RecSys maximális büntetés
            
        total_clicks += current_clicks

    print("\n" + "🔥"*15 + " EREDMÉNYEK (TISZTA W2V) " + "🔥"*15)
    for k in k_list:
        print(f"--- Top {k} Ajánlás ---")
        print(f"✨ Hit Rate:  {hits[k]/total:.4%}")
        print(f"📈 NDCG:      {ndcg[k]/total:.4%}")
        print(f"🎯 MAP:       {map_score[k]/total:.4%}\n")
    print("="*45)
    # Az átlagos kattintásszám:
    print(f"🖱️ Átlagos RecSys Clicks: {total_clicks/total:.4f}")
    print("="*45)

# Indítás a TELJES teszthalmazon (kivettem a [::10]-et!)
run_evaluation_multi_k(test_playlists[::10], k_list=[10, 20, 50, 100, 500])

📂 Adatok betöltése...
🏗️ Modell felépítése és súlyok betöltése...



c:\Users\Béres Gábor\music_recommender\.venv\Lib\site-packages\keras\src\layers\layer.py:982: UserWarning: Layer 'dot_scoring' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


✅ Kaggle W2V súlyok sikeresen betöltve!
✂️ Inferencia modell kivágása a jósláshoz...
⚡ Finomhangolt W2V vektorok kinyerése a modellből...
🏗️ FAISS index építése...
📊 FAISS Indexben tárolt dalok száma: 372433

🧐 Fair kiértékelés indítása 9,918 listán...


Evaluating:   0%|          | 0/9918 [00:00<?, ?it/s]


🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥 EREDMÉNYEK (TISZTA W2V) 🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
--- Top 10 Ajánlás ---
✨ Hit Rate:  8.8223%
📈 NDCG:      4.9684%
🎯 MAP:       3.8012%

--- Top 20 Ajánlás ---
✨ Hit Rate:  12.4824%
📈 NDCG:      5.8839%
🎯 MAP:       4.0473%

--- Top 50 Ajánlás ---
✨ Hit Rate:  18.9756%
📈 NDCG:      7.1657%
🎯 MAP:       4.2504%

--- Top 100 Ajánlás ---
✨ Hit Rate:  24.9244%
📈 NDCG:      8.1293%
🎯 MAP:       4.3349%

--- Top 500 Ajánlás ---
✨ Hit Rate:  43.7891%
📈 NDCG:      10.5487%
🎯 MAP:       4.4260%

🖱️ Átlagos RecSys Clicks: 34.0106


# Baseline (scratch) kiértékelése hibrid téren

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import faiss
import os
import gc
from tqdm.auto import tqdm

# ==========================================
# 1. ÚTVONALAK (Állítsd be a saját gépedhez!)
# ==========================================
BASE_DIR = "../Models" 

HYBRID_MATRIX_PATH = os.path.join(BASE_DIR, "hybrid_embedding_matrix.npy")
TEST_DATA_PATH = os.path.join(BASE_DIR, "test_pids.npy")
WEIGHTS_PATH = os.path.join(BASE_DIR, "best_sasrec_baseline_scratch.weights.h5")

# ==========================================
# 2. HIPERPARAMÉTEREK & ADATBETÖLTÉS
# ==========================================
MAX_LEN = 50

print("📂 Adatok betöltése...")
w2v_weights = np.load(HYBRID_MATRIX_PATH).astype('float32')
vocab_size = w2v_weights.shape[0]
del w2v_weights

D_MODEL = 256 
test_playlists = np.load(TEST_DATA_PATH, allow_pickle=True)

# ==========================================
# 3. AZ ÚJ, NAGY ARCHITEKTÚRA
# ==========================================
def create_large_bpr_model(vocab_size):
    seq_inputs = layers.Input(shape=(MAX_LEN,), name="seq_inputs")
    pos_inputs = layers.Input(shape=(MAX_LEN,), name="pos_inputs")
    neg_inputs = layers.Input(shape=(MAX_LEN,), name="neg_inputs")

    item_embedding = layers.Embedding(
        vocab_size, D_MODEL, mask_zero=True,
        trainable=True, name="item_embedding"
    )
    pos_embedding = layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")

    x = item_embedding(seq_inputs)
    x += pos_embedding(tf.range(MAX_LEN)[tf.newaxis, :])
    
    for i in range(3): 
        attn = layers.MultiHeadAttention(num_heads=8, key_dim=D_MODEL//8, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))
        
    seq_states = layers.Dense(D_MODEL, name="final_projection")(x)

    pos_emb = item_embedding(pos_inputs)
    neg_emb = item_embedding(neg_inputs)

    def dot_and_cast(tensors):
        seq, emb = tensors
        scores = tf.reduce_sum(seq * emb, axis=-1, keepdims=True)
        return tf.cast(scores, tf.float32)

    dot_layer = layers.Lambda(dot_and_cast, name="dot_scoring")
    
    pos_scores = dot_layer([seq_states, pos_emb])
    neg_scores = dot_layer([seq_states, neg_emb])

    outputs = layers.Concatenate(axis=-1, name="concat_scores")([pos_scores, neg_scores])
    return tf.keras.Model(inputs=[seq_inputs, pos_inputs, neg_inputs], outputs=outputs)

print("🏗️ Modell felépítése és súlyok betöltése...")
model = create_large_bpr_model(vocab_size)
model.load_weights(WEIGHTS_PATH)
print("✅ Kaggle W2V súlyok sikeresen betöltve!")

gc.collect()

# ==========================================
# 4. INFERENCIA & FAISS (JAVÍTVA)
# ==========================================
print("✂️ Inferencia modell kivágása a jósláshoz...")
inf_inputs = model.inputs[0] 
inf_outputs = model.get_layer("final_projection").output
inference_model = tf.keras.Model(inputs=inf_inputs, outputs=inf_outputs)

print("⚡ Finomhangolt W2V vektorok kinyerése a modellből...")
# <--- JAVÍTVA A RÉTEGNÉV!
updated_embeddings = model.get_layer("item_embedding").get_weights()[0] 

print("🏗️ FAISS index építése...")
faiss_embeddings = updated_embeddings.copy().astype('float32')
faiss.normalize_L2(faiss_embeddings)
index = faiss.IndexFlatIP(D_MODEL)
index.add(faiss_embeddings)
print(f"📊 FAISS Indexben tárolt dalok száma: {index.ntotal}")

# ==========================================
# 6. MULTI-K FAIR KIÉRTÉKELŐ (+ CLICKS)
# ==========================================
def run_evaluation_multi_k(playlists, k_list=[10, 50, 100]):
    max_k = max(k_list)
    hits = {k: 0 for k in k_list}
    ndcg = {k: 0 for k in k_list}
    map_score = {k: 0 for k in k_list}
    total_clicks = 0
    total = 0
    
    print(f"\n🧐 Fair kiértékelés indítása {len(playlists):,} listán...")
    
    for pl in tqdm(playlists, desc="Evaluating"):
        if len(pl) < 2: continue
        
        context_indices = pl[:-1] 
        target_id = pl[-1]
        
        input_seq = context_indices[-MAX_LEN:]
        pad_len = MAX_LEN - len(input_seq)
        input_padded = np.array([[0] * pad_len + input_seq])
        
        pred_vectors = inference_model.predict(input_padded, verbose=0)
        target_pred_vector = pred_vectors[0, -1, :].reshape(1, -1).astype('float32')
        
        faiss.normalize_L2(target_pred_vector)
        _, raw_top_indices = index.search(target_pred_vector, max_k + len(context_indices))
        raw_top_indices = raw_top_indices[0]
        
        recommendations = []
        for idx in raw_top_indices:
            if idx not in context_indices:
                recommendations.append(idx)
            if len(recommendations) == max_k:
                break
        
        total += 1
        
        # --- METRIKÁK ÉS CLICKS SZÁMÍTÁSA ---
        if target_id in recommendations:
            rank = recommendations.index(target_id) + 1
            current_clicks = (rank - 1) // 10  # RecSys Clicks matek
            
            for k in k_list:
                if rank <= k:
                    hits[k] += 1
                    ndcg[k] += 1 / np.log2(rank + 1)
                    map_score[k] += 1 / rank
        else:
            current_clicks = 51 # RecSys maximális büntetés
            
        total_clicks += current_clicks

    print("\n" + "🔥"*15 + " EREDMÉNYEK (TISZTA W2V) " + "🔥"*15)
    for k in k_list:
        print(f"--- Top {k} Ajánlás ---")
        print(f"✨ Hit Rate:  {hits[k]/total:.4%}")
        print(f"📈 NDCG:      {ndcg[k]/total:.4%}")
        print(f"🎯 MAP:       {map_score[k]/total:.4%}\n")
    print("="*45)
    # Az átlagos kattintásszám:
    print(f"🖱️ Átlagos RecSys Clicks: {total_clicks/total:.4f}")
    print("="*45)

# Indítás a TELJES teszthalmazon (kivettem a [::10]-et!)
run_evaluation_multi_k(test_playlists[::10], k_list=[1, 5, 10, 20, 50, 100, 200, 500])

📂 Adatok betöltése...
🏗️ Modell felépítése és súlyok betöltése...



c:\Users\Béres Gábor\music_recommender\.venv\Lib\site-packages\keras\src\layers\layer.py:982: UserWarning: Layer 'dot_scoring' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


✅ Kaggle W2V súlyok sikeresen betöltve!
✂️ Inferencia modell kivágása a jósláshoz...
⚡ Finomhangolt W2V vektorok kinyerése a modellből...
🏗️ FAISS index építése...
📊 FAISS Indexben tárolt dalok száma: 372433

🧐 Fair kiértékelés indítása 9,918 listán...


Evaluating:   0%|          | 0/9918 [00:00<?, ?it/s]


🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥 EREDMÉNYEK (TISZTA W2V) 🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
--- Top 1 Ajánlás ---
✨ Hit Rate:  0.2117%
📈 NDCG:      0.2117%
🎯 MAP:       0.2117%

--- Top 5 Ajánlás ---
✨ Hit Rate:  0.9881%
📈 NDCG:      0.5860%
🎯 MAP:       0.4559%

--- Top 10 Ajánlás ---
✨ Hit Rate:  1.6838%
📈 NDCG:      0.8090%
🎯 MAP:       0.5467%

--- Top 20 Ajánlás ---
✨ Hit Rate:  3.0349%
📈 NDCG:      1.1497%
🎯 MAP:       0.6398%

--- Top 50 Ajánlás ---
✨ Hit Rate:  5.6665%
📈 NDCG:      1.6623%
🎯 MAP:       0.7181%

--- Top 100 Ajánlás ---
✨ Hit Rate:  9.1853%
📈 NDCG:      2.2292%
🎯 MAP:       0.7669%

--- Top 200 Ajánlás ---
✨ Hit Rate:  14.3577%
📈 NDCG:      2.9499%
🎯 MAP:       0.8033%

--- Top 500 Ajánlás ---
✨ Hit Rate:  24.1278%
📈 NDCG:      4.1208%
🎯 MAP:       0.8342%

🖱️ Átlagos RecSys Clicks: 43.0228


# Vektortér módosítás meghatározása a trainable=True hatása miatt

In [3]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Eredeti W2V mátrix
original_w2v = np.load("../Models/word2vec_matrix.npy").astype('float32')

# 2. Trainable modell betöltése
model.load_weights("../Models/best_sasrec_large_hard_w2v_only.weights.h5")
trained = model.get_layer("w2v_embedding").get_weights()[0]

# 3. Drift mérés
sample_ids = np.random.choice(vocab_size, 5000)
similarities = np.diag(cosine_similarity(
    original_w2v[sample_ids], 
    trained[sample_ids]
))

print(f"Átlag:   {similarities.mean():.4f}")
print(f"Minimum: {similarities.min():.4f}")
print(f"Maximum: {similarities.max():.4f}")
print(f"Szórás:  {similarities.std():.4f}")

Átlag:   0.9981
Minimum: 0.9869
Maximum: 1.0000
Szórás:  0.0014


In [4]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. Eredeti W2V mátrix
original_w2v = np.load("../Models/word2vec_matrix.npy").astype('float32')

# 2. Trainable modell betöltése
model.load_weights("../Models/best_sasrec_large_hard_w2v_frozen.weights.h5")
trained = model.get_layer("w2v_embedding").get_weights()[0]

# 3. Drift mérés
sample_ids = np.random.choice(vocab_size, 5000)
similarities = np.diag(cosine_similarity(
    original_w2v[sample_ids], 
    trained[sample_ids]
))

print(f"Átlag:   {similarities.mean():.4f}")
print(f"Minimum: {similarities.min():.4f}")
print(f"Maximum: {similarities.max():.4f}")
print(f"Szórás:  {similarities.std():.4f}")

Átlag:   1.0000
Minimum: 1.0000
Maximum: 1.0000
Szórás:  0.0000


# Baseline, training from scratch tartalom alapú

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import faiss
import os
import gc
from tqdm.auto import tqdm

# ==========================================
# 1. ÚTVONALAK
# ==========================================
BASE_DIR = "../Models"

AE_MATRIX_PATH = os.path.join(BASE_DIR, "ae_vectors_closed_world_mel_only.npy")
TEST_DATA_PATH = os.path.join(BASE_DIR, "test_pids_ae.npy")
WEIGHTS_PATH = os.path.join(BASE_DIR, "best_sasrec_baseline_scratch_ae.weights.h5")

# ==========================================
# 2. HIPERPARAMÉTEREK & ADATBETÖLTÉS
# ==========================================
MAX_LEN = 50
D_MODEL = 256

print("📂 Adatok betöltése...")

# Vocab size az AE mátrix alapján (nincs W2V/hybrid mátrix a baseline-nál!)
ae_shape = np.load(AE_MATRIX_PATH, mmap_mode='r').shape
vocab_size = ae_shape[0] + 1  # +1 a 0-s padding sor miatt
print(f"📏 Szótár mérete: {vocab_size} dal")

test_playlists = np.load(TEST_DATA_PATH, allow_pickle=True)
print(f"🎵 Teszt lejátszási listák száma: {len(test_playlists):,}")

gc.collect()

# ==========================================
# 3. MODELL DEFINÍCIÓ (azonos a tanítóval!)
# ==========================================
def create_baseline_sasrec_model(vocab_size):
    seq_inputs = layers.Input(shape=(MAX_LEN,), name="seq_inputs")
    pos_inputs = layers.Input(shape=(MAX_LEN,), name="pos_inputs")
    neg_inputs = layers.Input(shape=(MAX_LEN,), name="neg_inputs")

    # FONTOS: a rétegnév "item_embedding_scratch" — egyeznie kell a tanítóval!
    item_embedding = layers.Embedding(
        vocab_size, D_MODEL, mask_zero=True,
        trainable=True,
        name="item_embedding_scratch"
    )
    pos_embedding = layers.Embedding(MAX_LEN, D_MODEL, name="pos_embedding")

    x = item_embedding(seq_inputs)
    x += pos_embedding(tf.range(MAX_LEN)[tf.newaxis, :])

    # 3 Transformer blokk, 8 fej (D_MODEL=256 → key_dim=32)
    for i in range(3):
        attn = layers.MultiHeadAttention(num_heads=8, key_dim=D_MODEL // 8, name=f"attn_{i}")(x, x, use_causal_mask=True)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(attn))
        ffn = layers.Dense(D_MODEL * 4, activation='relu')(x)
        ffn = layers.Dense(D_MODEL)(ffn)
        x = layers.LayerNormalization()(x + layers.Dropout(0.2)(ffn))

    seq_states = layers.Dense(D_MODEL, name="final_projection")(x)

    pos_emb = item_embedding(pos_inputs)
    neg_emb = item_embedding(neg_inputs)

    def dot_and_cast(tensors):
        seq, emb = tensors
        scores = tf.reduce_sum(seq * emb, axis=-1, keepdims=True)
        return tf.cast(scores, tf.float32)

    dot_layer = layers.Lambda(dot_and_cast, name="dot_scoring")

    pos_scores = dot_layer([seq_states, pos_emb])
    neg_scores = dot_layer([seq_states, neg_emb])

    outputs = layers.Concatenate(axis=-1, name="concat_scores")([pos_scores, neg_scores])
    return tf.keras.Model(inputs=[seq_inputs, pos_inputs, neg_inputs], outputs=outputs)

# ==========================================
# 4. MODELL FELÉPÍTÉSE ÉS SÚLYOK BETÖLTÉSE
# ==========================================
print("🏗️ Baseline SASRec modell felépítése...")
model = create_baseline_sasrec_model(vocab_size)
model.load_weights(WEIGHTS_PATH)
print("✅ Baseline súlyok sikeresen betöltve!")

gc.collect()

# ==========================================
# 5. INFERENCIA MODELL KIVÁGÁSA
# ==========================================
print("✂️ Inferencia modell kivágása a jósláshoz...")
inf_inputs = model.inputs[0]
inf_outputs = model.get_layer("final_projection").output
inference_model = tf.keras.Model(inputs=inf_inputs, outputs=inf_outputs)

# ==========================================
# 6. FAISS INDEX ÉPÍTÉSE
# ==========================================
print("⚡ Tanult embedding vektorok kinyerése a modellből...")
# FONTOS: "item_embedding_scratch" — NEM "item_embedding"!
updated_embeddings = model.get_layer("item_embedding_scratch").get_weights()[0]

print("🏗️ FAISS index építése...")
faiss_embeddings = updated_embeddings.copy().astype('float32')
faiss.normalize_L2(faiss_embeddings)
index = faiss.IndexFlatIP(D_MODEL)
index.add(faiss_embeddings)
print(f"📊 FAISS indexben tárolt dalok száma: {index.ntotal}")

gc.collect()

# ==========================================
# 7. MULTI-K FAIR KIÉRTÉKELŐ (+ CLICKS)
# ==========================================
def run_evaluation_multi_k(playlists, k_list=[10, 50, 100]):
    max_k = max(k_list)
    hits = {k: 0 for k in k_list}
    ndcg = {k: 0 for k in k_list}
    map_score = {k: 0 for k in k_list}
    total_clicks = 0
    total = 0

    print(f"\n🧐 Fair kiértékelés indítása {len(playlists):,} listán...")

    for pl in tqdm(playlists, desc="Evaluating"):
        if len(pl) < 2:
            continue

        context_indices = pl[:-1]
        target_id = pl[-1]

        input_seq = context_indices[-MAX_LEN:]
        pad_len = MAX_LEN - len(input_seq)
        input_padded = np.array([[0] * pad_len + list(input_seq)])

        pred_vectors = inference_model.predict(input_padded, verbose=0)
        target_pred_vector = pred_vectors[0, -1, :].reshape(1, -1).astype('float32')

        faiss.normalize_L2(target_pred_vector)
        _, raw_top_indices = index.search(target_pred_vector, max_k + len(context_indices))
        raw_top_indices = raw_top_indices[0]

        # Kontextusban már szereplő dalok kiszűrése
        recommendations = []
        for idx in raw_top_indices:
            if idx not in context_indices:
                recommendations.append(idx)
            if len(recommendations) == max_k:
                break

        total += 1

        # Metrikák és Clicks számítása
        if target_id in recommendations:
            rank = recommendations.index(target_id) + 1
            current_clicks = (rank - 1) // 10  # RecSys Clicks formula

            for k in k_list:
                if rank <= k:
                    hits[k] += 1
                    ndcg[k] += 1 / np.log2(rank + 1)
                    map_score[k] += 1 / rank
        else:
            current_clicks = 51  # RecSys maximális büntetés

        total_clicks += current_clicks

    print("\n" + "🔥" * 15 + " EREDMÉNYEK (BASELINE FROM SCRATCH) " + "🔥" * 15)
    for k in k_list:
        print(f"--- Top {k} Ajánlás ---")
        print(f"✨ Hit Rate:  {hits[k] / total:.4%}")
        print(f"📈 NDCG:      {ndcg[k] / total:.4%}")
        print(f"🎯 MAP:       {map_score[k] / total:.4%}\n")
    print("=" * 55)
    print(f"🖱️  Átlagos RecSys Clicks: {total_clicks / total:.4f}")
    print("=" * 55)


# ==========================================
# 8. KIÉRTÉKELÉS INDÍTÁSA
# ==========================================
# [::10] → minden 10. playlist a gyors futáshoz; távolítsd el a teljes kiértékeléshez
run_evaluation_multi_k(
    test_playlists[::10],
    k_list=[1, 5, 10, 20, 50, 100, 200, 500]
)

📂 Adatok betöltése...
📏 Szótár mérete: 27053 dal
🎵 Teszt lejátszási listák száma: 88,316
🏗️ Baseline SASRec modell felépítése...

✅ Baseline súlyok sikeresen betöltve!


c:\Users\Béres Gábor\music_recommender\.venv\Lib\site-packages\keras\src\layers\layer.py:982: UserWarning: Layer 'dot_scoring' (of type Lambda) was passed an input with a mask attached to it. However, this layer does not support masking and will therefore destroy the mask information. Downstream layers will not see the mask.
  warnings.warn(


✂️ Inferencia modell kivágása a jósláshoz...
⚡ Tanult embedding vektorok kinyerése a modellből...
🏗️ FAISS index építése...
📊 FAISS indexben tárolt dalok száma: 27053

🧐 Fair kiértékelés indítása 8,832 listán...


Evaluating:   0%|          | 0/8832 [00:00<?, ?it/s]


🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥 EREDMÉNYEK (BASELINE FROM SCRATCH) 🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥🔥
--- Top 1 Ajánlás ---
✨ Hit Rate:  1.9022%
📈 NDCG:      1.9022%
🎯 MAP:       1.9022%

--- Top 5 Ajánlás ---
✨ Hit Rate:  6.3179%
📈 NDCG:      4.1369%
🎯 MAP:       3.4222%

--- Top 10 Ajánlás ---
✨ Hit Rate:  10.8922%
📈 NDCG:      5.6025%
🎯 MAP:       4.0191%

--- Top 20 Ajánlás ---
✨ Hit Rate:  16.7799%
📈 NDCG:      7.0875%
🎯 MAP:       4.4248%

--- Top 50 Ajánlás ---
✨ Hit Rate:  28.2835%
📈 NDCG:      9.3608%
🎯 MAP:       4.7862%

--- Top 100 Ajánlás ---
✨ Hit Rate:  38.7455%
📈 NDCG:      11.0538%
🎯 MAP:       4.9342%

--- Top 200 Ajánlás ---
✨ Hit Rate:  50.5208%
📈 NDCG:      12.6990%
🎯 MAP:       5.0183%

--- Top 500 Ajánlás ---
✨ Hit Rate:  66.2251%
📈 NDCG:      14.5975%
🎯 MAP:       5.0704%

🖱️  Átlagos RecSys Clicks: 24.9588
